# 🚀 Kaggle Cloud Training & Extreme-Noise Benchmark: ConvNeXt-V2-Tiny Biometrics (224x224 + Teacher-Student KD)
### Two-Stage `convnextv2_base` -> `convnextv2_tiny` Knowledge Distillation, Photometric-Retinex Stem, Bilateral Symmetry Multi-Scale Fusion & Hierarchical 23-Bracket LDL
**Dataset**: Curated 8,331-face `224x224` Real-Human Golden Dataset (`train=6,845`, `val=722`, `test=764`, 13 dataset families, 23 fine age brackets, 6 ITA skin-tone groups, 0 cross-split leakage).

Key Architectural & Mathematical Innovations:
- **Two-Stage Teacher-Student Knowledge Distillation & Top-3 SWA Model Soup**: Distills `convnextv2_base` (89.25M params) into `convnextv2_tiny` (29.34M params) with interval-constrained LDL refinement (`refine_interval_ldl_with_teacher`) and confidence-gated missing-gender imputation.
- **PhotometricRetinexStem**: Dichromatic Lambertian-Specular decomposition + Multi-Scale Retinex ($\sigma=1.2, 2.5$) + CIE luminance reliability envelope $w(Y)=4Y(1-Y)$ + ConvNeXt-V2 GRN residual gating.
- **Stage 2 ($28\times 28$) & Stage 3 ($14\times 14$) Coordinate Attention + Bilateral Sagittal Symmetry Gates**: Cancels unilateral raking shadows while localizing epidermal micro-wrinkles and nasolabial structure.
- **Hierarchical 23-Interval + 101-Bin LDL Coupling**: Jointly predicts 23 fine age brackets (`0-2` … `96-100`) and the 101-bin continuous age simplex with Tikhonov Laplacian smoothness regularization.
- **Strict Stage-4 ($7\times 7$) Gradient Isolation**: `gender_head` and `reid_head` route strictly through Stage 4 global craniofacial morphology (`z4`).
- **Extreme Environmental Stress-Test Suite**: Evaluates holdout `test` across `clean` + 6 severe corruption regimes (`raking_directional_light`, `extreme_overexposure_glare`, `extreme_lowlight_sensor_noise`, `motion_blur_psf`, `partial_occlusion`, `extreme_compound_storm`).

In [ ]:
import os, sys, math, time, json
from pathlib import Path
import torch

!pip install -q timm pyarrow fastparquet
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
print(f'[Env] PyTorch {torch.__version__} | Timm {timm.__version__} | GPUs: {num_gpus}')
for g in range(num_gpus):
    print(f'  [GPU {g}] {torch.cuda.get_device_name(g)} ({torch.cuda.get_device_properties(g).total_memory / 1e9:.1f} GB)')
assert num_gpus >= 1, 'Expected at least 1 GPU for training!'


In [ ]:
import cv2
import numpy as np
import pandas as pd
from torch.utils.data import Dataset

print('✅ FaceGoldenDataset with 100% real-pixel verification, 23 fine age intervals, and gender_mask initialized.')


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
print('✅ PhotometricRetinexStem + MultiScaleBiometricFusion + ConvNeXt-V2-Tiny BiometricMultiTaskModel defined.')


In [ ]:
print('✅ UnifiedBiometricLoss with fine_interval_ldl_loss, ldl_tikhonov_smoothness_loss, photometric_consistency_loss & SupCon initialized.')


In [ ]:
with open('train_ddp.py', 'w', encoding='utf-8') as f:
    f.write('"""\nConvNeXt-V2-Tiny Biometric Multi-Task Training & Extreme-Noise Evaluation Script\nSupports both 1-GPU (Tesla P100 / T4) and Multi-GPU (Dual Tesla T4 DDP via torchrun).\n"""\nimport os, sys, math, time, json, random, hashlib, tarfile, copy\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nfrom pathlib import Path\nfrom typing import Optional, Dict, Any, List, Tuple\n\nimport cv2\ncv2.setNumThreads(0)\ncv2.ocl.setUseOpenCL(False)\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import Dataset, DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\nimport timm\n\n# ---------------------------------------------------------------------------\n# 1. Reproducibility & Distributed / Single-GPU Initialization\n# ---------------------------------------------------------------------------\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\ntorch.cuda.manual_seed_all(SEED)\n\nworld_size = int(os.environ.get("WORLD_SIZE", "1"))\nlocal_rank = int(os.environ.get("LOCAL_RANK", "0"))\nuse_ddp = world_size > 1\n\nif use_ddp:\n    dist.init_process_group(backend="nccl")\n    torch.cuda.set_device(local_rank)\n    device = torch.device(f"cuda:{local_rank}")\nelse:\n    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")\n\nis_main_process = (local_rank == 0)\nif is_main_process:\n    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\n    print(f"[Init] Mode: {\'DDP\' if use_ddp else \'Single-Process\'} | World Size: {world_size} | Device: {device} ({gpu_name})", flush=True)\n\n# ---------------------------------------------------------------------------\n# 2. 23 Fine Age Intervals (3-5 Years), Boundary-Calibrated LDL & Dimorphism\n# ---------------------------------------------------------------------------\nFINE_AGE_INTERVALS: Tuple[Tuple[str, int, int], ...] = (\n    ("0-2", 0, 2), ("3-5", 3, 5), ("6-8", 6, 8), ("9-11", 9, 11),\n    ("12-14", 12, 14), ("15-17", 15, 17), ("18-21", 18, 21), ("22-25", 22, 25),\n    ("26-30", 26, 30), ("31-35", 31, 35), ("36-40", 36, 40), ("41-45", 41, 45),\n    ("46-50", 46, 50), ("51-55", 51, 55), ("56-60", 56, 60), ("61-65", 61, 65),\n    ("66-70", 66, 70), ("71-75", 71, 75), ("76-80", 76, 80), ("81-85", 81, 85),\n    ("86-90", 86, 90), ("91-95", 91, 95), ("96-100", 96, 100),\n)\nNUM_FINE_INTERVALS = len(FINE_AGE_INTERVALS)\n\nAGE_COHORTS: Tuple[Tuple[str, int, int], ...] = (\n    ("Infant", 0, 2), ("Early_Childhood", 3, 6), ("Child", 7, 12),\n    ("Adolescent", 13, 17), ("Young_Adult", 18, 24), ("Prime_Adult", 25, 34),\n    ("Mid_Adult", 35, 44), ("Mature_Adult", 45, 54), ("Senior", 55, 64),\n    ("Geriatric", 65, 100),\n)\n\ndef get_fine_interval_for_age(age: float) -> Tuple[str, int, int, int]:\n    clamped = max(0.0, min(100.0, float(age)))\n    age_int = int(round(clamped))\n    for idx, (name, low, high) in enumerate(FINE_AGE_INTERVALS):\n        if low <= age_int <= high:\n            return name, idx, low, high\n    name, low, high = FINE_AGE_INTERVALS[-1]\n    return name, len(FINE_AGE_INTERVALS) - 1, low, high\n\ndef get_cohort_for_age(age: float) -> Tuple[str, int]:\n    age_c = max(0.0, min(100.0, float(age)))\n    for idx, (name, low, high) in enumerate(AGE_COHORTS):\n        if low <= age_c <= high + 0.999:\n            return name, idx\n    return AGE_COHORTS[-1][0], len(AGE_COHORTS) - 1\n\ndef calculate_sexual_dimorphism_weight(age: float) -> float:\n    age_c = max(0.0, min(100.0, float(age)))\n    if age_c <= 6.0: return 0.10\n    if age_c <= 12.0: return 0.10 + 0.15 * ((age_c - 6.0) / 6.0)\n    if age_c < 18.0: return 0.25 + 0.75 * ((age_c - 12.0) / 6.0)\n    if age_c <= 64.0: return 1.00\n    if age_c <= 85.0: return 1.00 - 0.15 * ((age_c - 64.0) / 21.0)\n    return 0.85\n\ndef compute_gaussian_ldl(mean_age: float, sigma: float = 2.5, num_bins: int = 101) -> np.ndarray:\n    mean_age = float(np.clip(mean_age, 0.0, num_bins - 1))\n    bins = np.arange(num_bins, dtype=np.float64)\n    sigma = max(float(sigma), 0.5)\n    K = float(num_bins - 1)\n    q_pos = np.exp(-0.5 * ((bins - mean_age) / sigma) ** 2)\n    q_neg_lower = np.exp(-0.5 * ((-bins - mean_age) / sigma) ** 2)\n    q_neg_upper = np.exp(-0.5 * ((2.0 * K - bins - mean_age) / sigma) ** 2)\n    q = q_pos.copy()\n    q[1:] += q_neg_lower[1:]\n    q[:-1] += q_neg_upper[:-1]\n    q = np.maximum(q, 1e-12)\n    q /= np.sum(q)\n    cur_mean = float(np.sum(bins * q))\n    if abs(cur_mean - mean_age) > 1e-4:\n        lam = 0.0\n        p = q.copy()\n        for _ in range(25):\n            exp_factor = np.exp(np.clip(lam * (bins - mean_age), -50.0, 50.0))\n            w = q * exp_factor\n            w_sum = np.sum(w)\n            if w_sum <= 1e-15: break\n            p = w / w_sum\n            err = np.sum(bins * p) - mean_age\n            if abs(err) <= 1e-6: break\n            var = np.sum(((bins - np.sum(bins * p)) ** 2) * p)\n            if var < 1e-8: break\n            lam -= err / var\n        q = p\n    q = np.maximum(q, 1e-12)\n    q /= np.sum(q)\n    return q.astype(np.float32)\n\ndef compute_interval_box_ldl(age_min: float, age_max: float, sigma: float = 1.5, num_bins: int = 101) -> np.ndarray:\n    a_low = float(np.clip(min(age_min, age_max), 0.0, num_bins - 1))\n    a_high = float(np.clip(max(age_min, age_max), 0.0, num_bins - 1))\n    if abs(a_high - a_low) < 1e-3:\n        return compute_gaussian_ldl(a_low, sigma=max(sigma, 1.5), num_bins=num_bins)\n    bins = np.arange(num_bins, dtype=np.float64)\n    sigma = max(float(sigma), 0.5)\n    sqrt2_sigma = math.sqrt(2.0) * sigma\n    low_edge, high_edge = a_low - 0.5, a_high + 0.5\n    K = float(num_bins - 1)\n    erf_vec = np.vectorize(math.erf)\n    q_main = 0.5 * (erf_vec((bins - low_edge) / sqrt2_sigma) - erf_vec((bins - high_edge) / sqrt2_sigma))\n    b_ref_low = -1.0 - bins\n    q_ref_low = 0.5 * (erf_vec((b_ref_low - low_edge) / sqrt2_sigma) - erf_vec((b_ref_low - high_edge) / sqrt2_sigma))\n    b_ref_high = 2.0 * K + 1.0 - bins\n    q_ref_high = 0.5 * (erf_vec((b_ref_high - low_edge) / sqrt2_sigma) - erf_vec((b_ref_high - high_edge) / sqrt2_sigma))\n    q = np.maximum(q_main + q_ref_low + q_ref_high, 1e-12)\n    q /= np.sum(q)\n    return q.astype(np.float32)\n\ndef integrate_101_to_fine_intervals(ldl_101: np.ndarray) -> np.ndarray:\n    probs = np.zeros(NUM_FINE_INTERVALS, dtype=np.float32)\n    for idx, (_, low, high) in enumerate(FINE_AGE_INTERVALS):\n        probs[idx] = float(np.sum(ldl_101[low : high + 1]))\n    s = float(np.sum(probs))\n    if s > 1e-12:\n        probs /= s\n    return probs\n\n# ---------------------------------------------------------------------------\n# 3. Physical Environmental Transforms & Extreme Noise Stress Benchmark\n# ---------------------------------------------------------------------------\ndef apply_overexposure_transform(image: np.ndarray, gamma: float = 0.5, flare_strength: float = 0.6, flare_center: Optional[Tuple[int, int]] = None) -> np.ndarray:\n    H, W = image.shape[:2]\n    gamma = float(np.clip(gamma, 0.2, 0.9))\n    lut = np.array([((i / 255.0) ** gamma) * 255.0 for i in range(256)], dtype=np.uint8)\n    img_gamma = cv2.LUT(image, lut)\n    if flare_center is None:\n        x0 = int(np.random.randint(int(W * 0.2), int(W * 0.8)))\n        y0 = int(np.random.randint(int(H * 0.1), int(H * 0.6)))\n    else:\n        x0, y0 = flare_center\n    sigma = float(np.random.uniform(15.0, 35.0))\n    Y, X = np.ogrid[:H, :W]\n    dist_sq = (X - x0) ** 2 + (Y - y0) ** 2\n    flare = np.exp(-dist_sq / (2.0 * sigma ** 2)) * (255.0 * flare_strength)\n    flare_3ch = np.dstack([flare, flare, flare])\n    img_flare = np.clip(img_gamma.astype(np.float32) + flare_3ch, 0.0, 255.0).astype(np.uint8)\n    lab = cv2.cvtColor(img_flare, cv2.COLOR_RGB2LAB)\n    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))\n    lab[:, :, 0] = clahe.apply(lab[:, :, 0])\n    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)\n\ndef apply_underexposure_backlight_transform(image: np.ndarray, gamma: float = 2.2, noise_sigma: float = 12.0, gradient_dir: str = "horizontal") -> np.ndarray:\n    H, W = image.shape[:2]\n    gamma = float(np.clip(gamma, 1.2, 3.5))\n    lut = np.array([((i / 255.0) ** gamma) * 255.0 for i in range(256)], dtype=np.uint8)\n    img_dark = cv2.LUT(image, lut).astype(np.float32)\n    if gradient_dir == "horizontal":\n        ramp = np.linspace(0.4, 1.0, W, dtype=np.float32).reshape(1, W, 1)\n    elif gradient_dir == "vertical":\n        ramp = np.linspace(1.2, 0.5, H, dtype=np.float32).reshape(H, 1, 1)\n    else:\n        ramp = 1.0\n    img_grad = img_dark * ramp\n    shot_noise = np.random.normal(0, 1.0, img_grad.shape).astype(np.float32) * np.sqrt(np.maximum(img_grad, 0.0) * 0.4)\n    read_noise = np.random.normal(0, noise_sigma, img_grad.shape).astype(np.float32)\n    return np.clip(img_grad + shot_noise + read_noise, 0.0, 255.0).astype(np.uint8)\n\n_NORMAL_CACHE: Dict[Tuple[int, int], Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = {}\n\ndef apply_raking_directional_lighting_transform(image: np.ndarray, azimuth_deg: float = 75.0, elevation_deg: float = 15.0, ambient: float = 0.18, direct_strength: float = 1.25, color_temp_shift: float = 0.0) -> np.ndarray:\n    H, W = image.shape[:2]\n    key = (H, W)\n    if key not in _NORMAL_CACHE:\n        v = np.linspace(-1.0, 1.0, H, dtype=np.float32).reshape(H, 1)\n        u = np.linspace(-1.0, 1.0, W, dtype=np.float32).reshape(1, W)\n        nx = 0.85 * u * np.exp(-0.35 * (u ** 2 + v ** 2))\n        ny = 0.75 * v * np.exp(-0.35 * (u ** 2 + v ** 2))\n        nose_env = np.exp(-(u ** 2) / (2.0 * 0.14 ** 2) - ((v - 0.02) ** 2) / (2.0 * 0.24 ** 2))\n        nx = nx + (u / (0.14 ** 2)) * 0.12 * nose_env\n        left_eye = np.exp(-((u + 0.32) ** 2 + (v + 0.16) ** 2) / (2.0 * 0.15 ** 2))\n        right_eye = np.exp(-((u - 0.32) ** 2 + (v + 0.16) ** 2) / (2.0 * 0.15 ** 2))\n        ny = ny - ((v + 0.16) / (0.15 ** 2)) * 0.05 * (left_eye + right_eye)\n        nz = np.ones_like(nx)\n        norm_len = np.sqrt(nx ** 2 + ny ** 2 + nz ** 2 + 1e-8)\n        nx, ny, nz = nx / norm_len, ny / norm_len, nz / norm_len\n        _NORMAL_CACHE[key] = (u, v, nx, ny, nz)\n    u, v, nx, ny, nz = _NORMAL_CACHE[key]\n    az_rad, el_rad = math.radians(float(azimuth_deg)), math.radians(float(elevation_deg))\n    lx, ly, lz = math.sin(az_rad) * math.cos(el_rad), -math.sin(el_rad), math.cos(az_rad) * math.cos(el_rad)\n    cos_theta = np.maximum(0.0, nx * lx + ny * ly + nz * lz)\n    shadow_mask = np.ones((H, W), dtype=np.float32)\n    if abs(azimuth_deg) > 40.0:\n        shadow_sign = -1.0 if azimuth_deg > 0 else 1.0\n        contra_u = np.maximum(0.0, shadow_sign * u)\n        cast_len = min(0.45, 0.12 + 0.30 * (abs(azimuth_deg) / 90.0))\n        nose_shadow = np.exp(-((contra_u - 0.18) ** 2) / (2.0 * (cast_len * 0.5) ** 2) - ((v - 0.04) ** 2) / (2.0 * 0.22 ** 2)) * (contra_u > 0.02).astype(np.float32)\n        shadow_mask = 1.0 - 0.65 * nose_shadow\n    shading = float(np.clip(ambient, 0.02, 0.8)) + float(direct_strength) * cos_theta * shadow_mask\n    shift = float(np.clip(color_temp_shift, -0.4, 0.4))\n    gains = np.array([1.0 - 0.55 * shift, 1.0 - 0.05 * abs(shift), 1.0 + 0.55 * shift], dtype=np.float32).reshape(1, 1, 3)\n    return np.clip(image.astype(np.float32) * shading[:, :, np.newaxis] * gains, 0.0, 255.0).astype(np.uint8)\n\ndef apply_perspective_pose_foreshortening_transform(image: np.ndarray, yaw_deg: float = 20.0, pitch_deg: float = 12.0, roll_deg: float = 5.0) -> np.ndarray:\n    H, W = image.shape[:2]\n    src_pts = np.array([[0.0, 0.0], [W - 1.0, 0.0], [W - 1.0, H - 1.0], [0.0, H - 1.0]], dtype=np.float32)\n    yaw_frac = float(np.clip(yaw_deg / 90.0, -0.35, 0.35)) * 0.22 * W\n    pitch_frac = float(np.clip(pitch_deg / 90.0, -0.35, 0.35)) * 0.18 * H\n    dst_pts = src_pts.copy()\n    if yaw_frac >= 0:\n        dst_pts[1, 0] -= yaw_frac; dst_pts[2, 0] -= yaw_frac\n        dst_pts[1, 1] += 0.35 * yaw_frac; dst_pts[2, 1] -= 0.35 * yaw_frac\n    else:\n        ay = abs(yaw_frac)\n        dst_pts[0, 0] += ay; dst_pts[3, 0] += ay\n        dst_pts[0, 1] += 0.35 * ay; dst_pts[3, 1] -= 0.35 * ay\n    if pitch_frac >= 0:\n        dst_pts[0, 0] += 0.30 * pitch_frac; dst_pts[1, 0] -= 0.30 * pitch_frac\n        dst_pts[0, 1] += pitch_frac; dst_pts[1, 1] += pitch_frac\n    else:\n        ap = abs(pitch_frac)\n        dst_pts[3, 0] += 0.30 * ap; dst_pts[2, 0] -= 0.30 * ap\n        dst_pts[3, 1] -= ap; dst_pts[2, 1] -= ap\n    H_mat = cv2.getPerspectiveTransform(src_pts, dst_pts)\n    warped = cv2.warpPerspective(image, H_mat, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)\n    if abs(roll_deg) > 0.5:\n        rot_mat = cv2.getRotationMatrix2D((W / 2.0, H / 2.0), float(roll_deg), 1.0)\n        warped = cv2.warpAffine(warped, rot_mat, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)\n    return warped\n\ndef apply_motion_blur_transform(image: np.ndarray, kernel_size: int = 7, angle_deg: float = 0.0) -> np.ndarray:\n    k = max(3, int(kernel_size))\n    if k % 2 == 0: k += 1\n    kernel = np.zeros((k, k), dtype=np.float32)\n    kernel[k // 2, :] = 1.0\n    rot_mat = cv2.getRotationMatrix2D((k / 2.0 - 0.5, k / 2.0 - 0.5), angle_deg, 1.0)\n    kernel = cv2.warpAffine(kernel, rot_mat, (k, k))\n    k_sum = kernel.sum()\n    if k_sum > 0: kernel /= k_sum\n    else: kernel[k // 2, k // 2] = 1.0\n    return cv2.filter2D(image, -1, kernel)\n\ndef apply_isp_degradation_and_compression_transform(image: np.ndarray, downscale_factor: float = 0.45, jpeg_quality: int = 22, chroma_noise_sigma: float = 6.0) -> np.ndarray:\n    H, W = image.shape[:2]\n    scale = float(np.clip(downscale_factor, 0.20, 1.0))\n    down = cv2.resize(image, (max(16, int(round(W * scale))), max(16, int(round(H * scale)))), interpolation=cv2.INTER_AREA)\n    up = cv2.resize(down, (W, H), interpolation=cv2.INTER_LINEAR)\n    if chroma_noise_sigma > 0.0:\n        noise = np.random.normal(0.0, float(chroma_noise_sigma), up.shape).astype(np.float32)\n        up = np.clip(up.astype(np.float32) + noise, 0.0, 255.0).astype(np.uint8)\n    ok, enc = cv2.imencode(".jpg", up, [int(cv2.IMWRITE_JPEG_QUALITY), int(np.clip(jpeg_quality, 8, 95))])\n    if ok:\n        dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n        if dec is not None and dec.shape == image.shape:\n            return dec\n    return up\n\ndef apply_cutout_occlusion_transform(image: np.ndarray, occlusion_type: str = "mask") -> np.ndarray:\n    out = image.copy()\n    H, W = out.shape[:2]\n    t = random.choice(["mask", "sunglasses"]) if occlusion_type == "random" else occlusion_type\n    if t == "mask":\n        poly = np.array([[int(W * 0.15), int(H * 0.58)], [int(W * 0.85), int(H * 0.58)], [int(W * 0.82), int(H * 0.96)], [int(W * 0.18), int(H * 0.96)]], dtype=np.int32)\n        cv2.fillPoly(out, [poly], [140, 180, 215])\n        cv2.line(out, (int(W * 0.18), int(H * 0.70)), (int(W * 0.82), int(H * 0.70)), (120, 160, 195), 1)\n    else:\n        lc, rc = (int(W * 0.35), int(H * 0.42)), (int(W * 0.65), int(H * 0.42))\n        axes = (int(W * 0.18), int(H * 0.12))\n        cv2.ellipse(out, lc, axes, 0, 0, 360, [20, 20, 25], -1)\n        cv2.ellipse(out, rc, axes, 0, 0, 360, [20, 20, 25], -1)\n        cv2.line(out, lc, rc, (30, 30, 35), 3)\n    return out\n\nclass PhysicalEnvironmentalAugmenter:\n    def __init__(self, p_overexposed=0.22, p_underexposed=0.22, p_raking_light=0.28, p_pose_warp=0.22, p_blur=0.22, p_isp_degrade=0.20, p_cutout=0.18):\n        self.p_overexposed, self.p_underexposed, self.p_raking_light = p_overexposed, p_underexposed, p_raking_light\n        self.p_pose_warp, self.p_blur, self.p_isp_degrade, self.p_cutout = p_pose_warp, p_blur, p_isp_degrade, p_cutout\n\n    def __call__(self, image_rgb: np.ndarray) -> np.ndarray:\n        out = image_rgb.copy()\n        if random.random() < 0.5:\n            out = cv2.flip(out, 1)\n        if random.random() < self.p_pose_warp:\n            out = apply_perspective_pose_foreshortening_transform(out, yaw_deg=random.uniform(-22.0, 22.0), pitch_deg=random.uniform(-14.0, 14.0), roll_deg=random.uniform(-7.0, 7.0))\n        r = random.random()\n        if r < self.p_overexposed:\n            out = apply_overexposure_transform(out, gamma=random.uniform(0.42, 0.72), flare_strength=random.uniform(0.35, 0.65))\n        elif r < self.p_overexposed + self.p_underexposed:\n            out = apply_underexposure_backlight_transform(out, gamma=random.uniform(1.75, 2.45), noise_sigma=random.uniform(6.0, 14.0))\n        elif r < self.p_overexposed + self.p_underexposed + self.p_raking_light:\n            az = random.choice([random.uniform(-82.0, -42.0), random.uniform(42.0, 82.0)])\n            out = apply_raking_directional_lighting_transform(out, azimuth_deg=az, elevation_deg=random.uniform(-30.0, 40.0), ambient=random.uniform(0.12, 0.28), color_temp_shift=random.uniform(-0.25, 0.25))\n        if random.random() < self.p_blur:\n            out = apply_motion_blur_transform(out, kernel_size=random.choice([3, 5, 7, 9]), angle_deg=random.uniform(0.0, 180.0))\n        if random.random() < self.p_isp_degrade:\n            out = apply_isp_degradation_and_compression_transform(out, downscale_factor=random.uniform(0.42, 0.72), jpeg_quality=random.randint(18, 42), chroma_noise_sigma=random.uniform(2.0, 6.0))\n        if random.random() < self.p_cutout:\n            out = apply_cutout_occlusion_transform(out, occlusion_type="random")\n        return out\n\nclass ExtremeNoiseStressBenchmark:\n    REGIMES = (\n        "clean",\n        "raking_directional_light",\n        "extreme_overexposure_glare",\n        "extreme_lowlight_sensor_noise",\n        "motion_blur_psf",\n        "partial_occlusion",\n        "extreme_compound_storm",\n    )\n\n    @classmethod\n    def apply_regime_uint8(cls, image_rgb: np.ndarray, regime: str, seed: int = 42) -> np.ndarray:\n        if regime == "clean":\n            return image_rgb.copy()\n        H, W = image_rgb.shape[:2]\n        rng_state = np.random.get_state()\n        py_state = random.getstate()\n        try:\n            np.random.seed(seed)\n            random.seed(seed)\n            if regime == "raking_directional_light":\n                az = 78.0 if (seed % 2 == 0) else -78.0\n                el = 22.0 if (seed % 3 != 0) else -20.0\n                return apply_raking_directional_lighting_transform(image_rgb, azimuth_deg=az, elevation_deg=el, ambient=0.14, direct_strength=1.35, color_temp_shift=0.22 if (seed % 2 == 0) else -0.22)\n            if regime == "extreme_overexposure_glare":\n                return apply_overexposure_transform(image_rgb, gamma=0.45, flare_strength=0.65, flare_center=(int(W * 0.55), int(H * 0.35)))\n            if regime == "extreme_lowlight_sensor_noise":\n                return apply_underexposure_backlight_transform(image_rgb, gamma=2.35, noise_sigma=14.0, gradient_dir="horizontal")\n            if regime == "motion_blur_psf":\n                warped = apply_perspective_pose_foreshortening_transform(image_rgb, yaw_deg=18.0, pitch_deg=10.0, roll_deg=4.0)\n                return apply_motion_blur_transform(warped, kernel_size=9, angle_deg=35.0)\n            if regime == "partial_occlusion":\n                return apply_cutout_occlusion_transform(image_rgb, occlusion_type="mask" if (seed % 2 == 0) else "sunglasses")\n            if regime == "extreme_compound_storm":\n                lit = apply_raking_directional_lighting_transform(image_rgb, azimuth_deg=-72.0, elevation_deg=18.0, ambient=0.16, direct_strength=1.28, color_temp_shift=-0.18)\n                blurred = apply_motion_blur_transform(lit, kernel_size=7, angle_deg=25.0)\n                return apply_isp_degradation_and_compression_transform(blurred, downscale_factor=0.48, jpeg_quality=24, chroma_noise_sigma=5.0)\n            raise ValueError(f"Unknown regime: {regime}")\n        finally:\n            np.random.set_state(rng_state)\n            random.setstate(py_state)\n\n# ---------------------------------------------------------------------------\n# 4. Curated Golden Dataset Loader with In-RAM / /dev/shm Pre-Caching\n# ---------------------------------------------------------------------------\ndef discover_golden_manifest() -> Path:\n    candidates = [\n        Path("/kaggle/input/notebooks/thanhmuefatty/face-dataset-mining-extreme-filtering-pipeline/golden_dataset/metadata.parquet"),\n        Path("/kaggle/input/face-dataset-mining-extreme-filtering-pipeline/golden_dataset/metadata.parquet"),\n        Path("/kaggle/input/notebooks/vnthanhphm/face-dataset-mining-extreme-filtering-pipeline/golden_dataset/metadata.parquet"),\n        Path("/kaggle/input/face-gender-age-golden-dataset/golden_dataset/metadata.parquet"),\n        Path("/kaggle/input/face-gender-age-golden-dataset/metadata.parquet"),\n        Path("reports/golden_dataset/metadata.parquet"),\n        Path("reports/metadata.parquet"),\n    ]\n    for c in candidates:\n        if c.exists():\n            return c\n    kaggle_in = Path("/kaggle/input")\n    if kaggle_in.exists():\n        found = sorted(kaggle_in.rglob("metadata.parquet"))\n        if found:\n            return found[0]\n    raise FileNotFoundError("Could not locate golden_dataset/metadata.parquet in /kaggle/input!")\n\n_SHARED_TAR_BYTES: Dict[str, bytes] = {}\n_SHARED_DF_ALL: Optional[pd.DataFrame] = None\n\nclass FaceGoldenDataset(Dataset):\n    def __init__(self, manifest_path: Path, split: str = "train", is_train: bool = False):\n        global _SHARED_DF_ALL\n        self.manifest_path = Path(manifest_path)\n        self.base_dir = self.manifest_path.parent\n        self.split = split\n        self.is_train = is_train\n        self.env_augmenter = PhysicalEnvironmentalAugmenter() if is_train else None\n\n        if _SHARED_DF_ALL is None:\n            _SHARED_DF_ALL = pd.read_parquet(self.manifest_path) if self.manifest_path.suffix == ".parquet" else pd.read_csv(self.manifest_path)\n        df_all = _SHARED_DF_ALL\n        if split and "split" in df_all.columns:\n            self.df = df_all[df_all["split"] == split].reset_index(drop=True)\n        else:\n            self.df = df_all.reset_index(drop=True)\n\n        # Index TAR shards once globally\n        if not _SHARED_TAR_BYTES:\n            shards_dir = self.base_dir / "shards"\n            if shards_dir.is_dir():\n                for sp in sorted(shards_dir.glob("shard_*.tar")):\n                    with tarfile.open(sp, "r") as tf:\n                        for member in tf.getmembers():\n                            if member.isfile():\n                                fobj = tf.extractfile(member)\n                                if fobj is not None:\n                                    raw = fobj.read()\n                                    name = Path(member.name).name\n                                    stem = Path(member.name).stem\n                                    _SHARED_TAR_BYTES[name] = raw\n                                    _SHARED_TAR_BYTES[stem] = raw\n        self._tar_bytes = _SHARED_TAR_BYTES\n\n        # Pre-decode all split images into contiguous (N, 224, 224, 3) uint8 RGB array in RAM\n        # and pre-compute all static target tensors once\n        N = len(self.df)\n        self.images_rgb = np.zeros((N, 224, 224, 3), dtype=np.uint8)\n        self.cached_targets: List[Dict[str, Any]] = []\n        fine_cols = [f"fine_ldl_{iv_name.replace(\'-\', \'_\')}" for iv_name, _, _ in FINE_AGE_INTERVALS]\n        has_all_fine_cols = all(c in self.df.columns for c in fine_cols)\n        decoded_ok = 0\n\n        for i in range(N):\n            row = self.df.iloc[i]\n            img_bgr = self._decode_row_bgr(row)\n            if img_bgr is not None and img_bgr.size > 0:\n                if img_bgr.shape[:2] != (224, 224):\n                    img_bgr = cv2.resize(img_bgr, (224, 224), interpolation=cv2.INTER_CUBIC)\n                self.images_rgb[i] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)\n                if int(self.images_rgb[i].max()) > 0:\n                    decoded_ok += 1\n\n            raw_age = row.get("age_continuous")\n            has_valid_age = 1.0 if (raw_age is not None and pd.notna(raw_age) and float(raw_age) >= 0.0) else 0.0\n            if "has_valid_age" in row and pd.notna(row.get("has_valid_age")):\n                if str(row.get("has_valid_age")).strip().lower() in {"false", "0", "0.0", "no"}:\n                    has_valid_age = 0.0\n            age_val = float(np.clip(float(raw_age), 0.0, 100.0)) if has_valid_age > 0.5 else 25.0\n            age_min = float(row.get("age_min")) if ("age_min" in row and pd.notna(row.get("age_min"))) else age_val\n            age_max = float(row.get("age_max")) if ("age_max" in row and pd.notna(row.get("age_max"))) else age_val\n            sigma_val = float(row.get("apparent_age_std", 2.5)) if ("apparent_age_std" in row and pd.notna(row.get("apparent_age_std"))) else 2.5\n            if math.isnan(sigma_val) or sigma_val <= 0: sigma_val = 2.5\n\n            ltype_str = str(row.get("age_label_type", "")).strip().lower() if ("age_label_type" in row and pd.notna(row.get("age_label_type"))) else ""\n            is_exact_label = ltype_str in {"exact", "fine_interval"}\n            if abs(age_max - age_min) > 1e-3 and not is_exact_label:\n                ldl_101 = compute_interval_box_ldl(age_min, age_max, sigma=sigma_val)\n            else:\n                ldl_101 = compute_gaussian_ldl(age_val, sigma=sigma_val)\n\n            if has_all_fine_cols:\n                vals_23 = np.array([float(row[c]) if pd.notna(row[c]) else 0.0 for c in fine_cols], dtype=np.float32)\n                s_23 = float(np.sum(vals_23))\n                ldl_fine_23 = (vals_23 / s_23) if (np.all(np.isfinite(vals_23)) and s_23 > 0.0) else integrate_101_to_fine_intervals(ldl_101)\n            else:\n                ldl_fine_23 = integrate_101_to_fine_intervals(ldl_101)\n\n            _, iv_idx, _, _ = get_fine_interval_for_age(age_val)\n\n            g_raw = row.get("gender")\n            has_gender = (g_raw is not None) and pd.notna(g_raw) and (str(g_raw).strip().lower() in ["female", "male", "f", "m", "1", "0", "1.0", "0.0"])\n            gender_mask = 1.0 if has_gender else 0.0\n            gender_target = 1.0 if (has_gender and str(g_raw).strip().lower() in ["female", "f", "1", "1.0"]) else 0.0\n\n            raw_dim = row.get("dimorphism_weight", calculate_sexual_dimorphism_weight(age_val))\n            dimorphism_w = float(raw_dim) if (raw_dim is not None and pd.notna(raw_dim) and not math.isnan(float(raw_dim))) else calculate_sexual_dimorphism_weight(age_val)\n\n            raw_alpha = row.get("alpha_sample_weight", 1.0)\n            alpha_w = float(raw_alpha) if (raw_alpha is not None and pd.notna(raw_alpha) and not math.isnan(float(raw_alpha))) else 1.0\n\n            identity_val = -1\n            for id_col in ["subject_id", "identity_id", "identity", "person_id"]:\n                if id_col in row and pd.notna(row.get(id_col)):\n                    sval = str(row.get(id_col)).strip()\n                    if not sval or sval.lower() in {"-1", "-1.0", "none", "nan", "null", "unknown"} or "unknown" in sval.lower():\n                        continue\n                    try:\n                        ival = int(float(sval))\n                        if ival >= 0:\n                            identity_val = ival\n                            break\n                    except (ValueError, TypeError):\n                        digest = hashlib.sha256(sval.encode("utf-8")).digest()\n                        identity_val = int.from_bytes(digest[:8], "big") & ((1 << 52) - 1)\n                        break\n\n            self.cached_targets.append({\n                "sample_idx": torch.tensor(i, dtype=torch.long),\n                "ldl_101": torch.from_numpy(ldl_101),\n                "ldl_fine_23": torch.from_numpy(ldl_fine_23),\n                "age_continuous": torch.tensor(age_val, dtype=torch.float32),\n                "age_min": torch.tensor(age_min, dtype=torch.float32),\n                "age_max": torch.tensor(age_max, dtype=torch.float32),\n                "age_interval_idx": torch.tensor(iv_idx, dtype=torch.long),\n                "has_valid_age": torch.tensor(has_valid_age, dtype=torch.float32),\n                "gender": torch.tensor(gender_target, dtype=torch.float32),\n                "gender_mask": torch.tensor(gender_mask, dtype=torch.float32),\n                "dimorphism_weight": torch.tensor(dimorphism_w, dtype=torch.float32),\n                "alpha_sample_weight": torch.tensor(alpha_w, dtype=torch.float32),\n                "identity": torch.tensor(identity_val, dtype=torch.long),\n                "ita_group": str(row.get("ita_group", row.get("skin_tone_group", "unknown"))),\n                "dataset_source": str(row.get("source_dataset", row.get("dataset_name", row.get("dataset_source", row.get("source", row.get("dataset", "unknown")))))),\n            })\n\n        self.decoded_ok = decoded_ok\n        if is_main_process:\n            print(f"[Dataset {split}] Loaded {N} samples (224x224) | Decoded non-zero RGB: {decoded_ok}/{N} ({100.0 * decoded_ok / max(1, N):.2f}%)", flush=True)\n        assert decoded_ok == N and N > 0, f"Dataset {split} failed 100% real image verification: {decoded_ok}/{N}"\n\n    def _decode_row_bgr(self, row: pd.Series) -> Optional[np.ndarray]:\n        img_id = str(row.get("image_id", ""))\n        if img_id in self._tar_bytes:\n            arr = np.frombuffer(self._tar_bytes[img_id], dtype=np.uint8)\n            return cv2.imdecode(arr, cv2.IMREAD_COLOR)\n        stem = Path(img_id).stem\n        if stem in self._tar_bytes:\n            arr = np.frombuffer(self._tar_bytes[stem], dtype=np.uint8)\n            return cv2.imdecode(arr, cv2.IMREAD_COLOR)\n        for k in ["image_path", "crop_path", "filepath", "path"]:\n            p = row.get(k)\n            if p and pd.notna(p) and os.path.exists(str(p)):\n                return cv2.imread(str(p))\n        return None\n\n    def __len__(self) -> int:\n        return len(self.df)\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        img_rgb = self.images_rgb[idx]\n        if self.is_train:\n            clean_rgb = img_rgb.copy()\n            if random.random() < 0.5:\n                clean_rgb = cv2.flip(clean_rgb, 1)\n            corrupt_rgb = self.env_augmenter(img_rgb)\n            tensor_clean = torch.from_numpy(clean_rgb.transpose(2, 0, 1)).float() / 127.5 - 1.0\n            tensor_corrupt = torch.from_numpy(corrupt_rgb.transpose(2, 0, 1)).float() / 127.5 - 1.0\n        else:\n            tensor_clean = torch.from_numpy(img_rgb.transpose(2, 0, 1)).float() / 127.5 - 1.0\n            tensor_corrupt = tensor_clean\n\n        out = dict(self.cached_targets[idx])\n        out["image"] = tensor_clean\n        out["image_corrupted"] = tensor_corrupt\n        return out\n\n# ---------------------------------------------------------------------------\n# 5. Bespoke Photometric-Retinex Multi-Scale ConvNeXt-V2 Architecture\n# ---------------------------------------------------------------------------\nclass GlobalResponseNorm2d(nn.Module):\n    def __init__(self, num_channels: int, eps: float = 1e-6):\n        super().__init__()\n        self.eps = float(eps)\n        self.gamma = nn.Parameter(torch.zeros(1, num_channels, 1, 1))\n        self.beta = nn.Parameter(torch.zeros(1, num_channels, 1, 1))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        orig_dtype = x.dtype\n        x_f = x.float()\n        gx = torch.sqrt(torch.mean(x_f * x_f, dim=(2, 3), keepdim=True) + self.eps)\n        nx = gx / (torch.mean(gx, dim=1, keepdim=True) + self.eps)\n        out = self.gamma.float() * (x_f * nx) + self.beta.float() + x_f\n        return out.to(orig_dtype)\n\nclass GlobalResponseNorm1d(nn.Module):\n    def __init__(self, dim: int, eps: float = 1e-6):\n        super().__init__()\n        self.eps = float(eps)\n        self.gamma = nn.Parameter(torch.zeros(1, dim))\n        self.beta = nn.Parameter(torch.zeros(1, dim))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        orig_dtype = x.dtype\n        x_f = x.float()\n        gx = torch.abs(x_f)\n        nx = gx / (torch.mean(gx, dim=-1, keepdim=True) + self.eps)\n        out = self.gamma.float() * (x_f * nx) + self.beta.float() + x_f\n        return out.to(orig_dtype)\n\nclass PhotometricRetinexStem(nn.Module):\n    """\n    Dichromatic Lambertian-Specular & Multi-Scale Retinex Normalizer Stem.\n    Extracts intrinsic log-reflectance at fine (sigma=1.2) and mesoscopic (sigma=2.5)\n    scales + CIE luminance exposure reliability envelope w(Y) = 4Y(1-Y), refined\n    through a ConvNeXt-V2 GRN residual block gated by tanh(alpha).\n    """\n    def __init__(self, hidden_channels: int = 24, eps: float = 1e-4):\n        super().__init__()\n        self.eps = float(eps)\n        self.register_buffer("kernel_fine", self._build_gaussian_kernel(kernel_size=5, sigma=1.2))\n        self.register_buffer("kernel_coarse", self._build_gaussian_kernel(kernel_size=9, sigma=2.5))\n        self.register_buffer("luma_weights", torch.tensor([0.299, 0.587, 0.114], dtype=torch.float32).view(1, 3, 1, 1))\n        self.dw_conv = nn.Conv2d(10, 10, kernel_size=5, padding=2, groups=10, bias=False)\n        self.norm = nn.GroupNorm(num_groups=2, num_channels=10)\n        self.pw_expand = nn.Conv2d(10, hidden_channels, kernel_size=1)\n        self.act = nn.GELU()\n        self.grn = GlobalResponseNorm2d(hidden_channels)\n        self.pw_proj = nn.Conv2d(hidden_channels, 3, kernel_size=1)\n        self.residual_gate = nn.Parameter(torch.tensor(0.15, dtype=torch.float32))\n\n    @staticmethod\n    def _build_gaussian_kernel(kernel_size: int, sigma: float) -> torch.Tensor:\n        coords = torch.arange(kernel_size, dtype=torch.float32) - (kernel_size - 1) / 2.0\n        g1d = torch.exp(-(coords ** 2) / (2.0 * (sigma ** 2)))\n        g2d = torch.outer(g1d, g1d)\n        g2d = g2d / torch.sum(g2d)\n        return g2d.view(1, 1, kernel_size, kernel_size).repeat(3, 1, 1, 1)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if x.ndim != 4 or x.shape[1] != 3:\n            return x\n        orig_dtype = x.dtype\n        with torch.cuda.amp.autocast(enabled=False):\n            x_f = x.float()\n            img_01 = torch.clamp((x_f + 1.0) * 0.5, min=self.eps, max=1.0)\n            log_i = torch.log(img_01 + self.eps)\n            pad_f = self.kernel_fine.shape[-1] // 2\n            pad_c = self.kernel_coarse.shape[-1] // 2\n            h_in, w_in = img_01.shape[2], img_01.shape[3]\n            mode_f = "reflect" if (h_in > pad_f and w_in > pad_f) else "replicate"\n            mode_c = "reflect" if (h_in > pad_c and w_in > pad_c) else "replicate"\n            illum_fine = F.conv2d(F.pad(img_01, (pad_f, pad_f, pad_f, pad_f), mode=mode_f), self.kernel_fine.float(), groups=3)\n            r_fine = log_i - torch.log(torch.clamp(illum_fine, min=self.eps))\n            illum_coarse = F.conv2d(F.pad(img_01, (pad_c, pad_c, pad_c, pad_c), mode=mode_c), self.kernel_coarse.float(), groups=3)\n            r_coarse = log_i - torch.log(torch.clamp(illum_coarse, min=self.eps))\n            luma = torch.sum(img_01 * self.luma_weights.float(), dim=1, keepdim=True)\n            exposure_weight = 4.0 * luma * (1.0 - luma)\n            feat_in = torch.cat([x_f, r_fine, r_coarse, exposure_weight], dim=1).to(dtype=self.dw_conv.weight.dtype)\n            delta = self.pw_proj(self.grn(self.act(self.pw_expand(self.norm(self.dw_conv(feat_in)))))).float()\n            out = torch.clamp(x_f + torch.tanh(self.residual_gate.float()) * delta, min=-2.5, max=2.5)\n        return out.to(orig_dtype)\n\nclass CoordinateAttention(nn.Module):\n    def __init__(self, in_channels: int, reduction: int = 16):\n        super().__init__()\n        mip = max(8, in_channels // reduction)\n        self.conv1 = nn.Conv2d(in_channels, mip, kernel_size=1, stride=1, padding=0, bias=False)\n        self.bn1 = nn.BatchNorm2d(mip)\n        self.act = nn.SiLU()\n        self.conv_h = nn.Conv2d(mip, in_channels, kernel_size=1, stride=1, padding=0, bias=False)\n        self.conv_w = nn.Conv2d(mip, in_channels, kernel_size=1, stride=1, padding=0, bias=False)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if x.ndim != 4 or x.shape[2] <= 1 or x.shape[3] <= 1:\n            return x\n        b, c, h, w = x.shape\n        x_h = torch.mean(x, dim=3, keepdim=True)\n        x_w = torch.mean(x, dim=2, keepdim=True).permute(0, 1, 3, 2)\n        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))\n        x_h_out, x_w_out = y[:, :, :h, :], y[:, :, h:, :].permute(0, 1, 3, 2)\n        return x * torch.sigmoid(self.conv_h(x_h_out)) * torch.sigmoid(self.conv_w(x_w_out))\n\nclass MultiScaleBiometricFusion(nn.Module):\n    """\n    Fuses Stage 2 (28x28 at 224x224), Stage 3 (14x14), and Stage 4 (7x7) with:\n    - Coordinate Attention on Stages 2 & 3\n    - Bilateral Sagittal Hemiface Symmetry Gates on Stages 2 & 3\n    - ConvNeXt-V2 GlobalResponseNorm1d residual fusion bottleneck\n    - Strict Stage-4 isolation for z4 (gender & Re-ID heads)\n    """\n    def __init__(self, in_channels: Tuple[int, int, int] = (192, 384, 768), proj_dim: int = 128, fused_dim: int = 512):\n        super().__init__()\n        c2, c3, c4 = in_channels\n        self.coord_att2 = CoordinateAttention(in_channels=c2, reduction=16)\n        self.coord_att3 = CoordinateAttention(in_channels=c3, reduction=16)\n        self.sym_gate2 = nn.Parameter(torch.zeros(1, c2, 1, 1))\n        self.sym_gate3 = nn.Parameter(torch.zeros(1, c3, 1, 1))\n        self.proj2 = nn.Conv2d(c2, proj_dim, kernel_size=1, bias=False)\n        self.proj3 = nn.Conv2d(c3, proj_dim, kernel_size=1, bias=False)\n        self.proj4 = nn.Conv2d(c4, proj_dim, kernel_size=1, bias=False)\n        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))\n        self.max_pool = nn.AdaptiveMaxPool2d((1, 1))\n        concat_dim = proj_dim * 2 * 3\n        self.fusion = nn.Sequential(nn.Linear(concat_dim, fused_dim), nn.LayerNorm(fused_dim))\n        self.fusion_grn_block = nn.Sequential(\n            nn.Linear(fused_dim, fused_dim),\n            nn.GELU(),\n            GlobalResponseNorm1d(fused_dim),\n            nn.Linear(fused_dim, fused_dim),\n        )\n        self.fusion_norm = nn.LayerNorm(fused_dim)\n\n    @staticmethod\n    def _apply_bilateral_symmetry(feat: torch.Tensor, gate: torch.Tensor) -> torch.Tensor:\n        if feat.ndim != 4 or feat.shape[3] <= 1:\n            return feat\n        feat_mirror = torch.flip(feat, dims=[3])\n        diff = 0.5 * (feat_mirror - feat)\n        scale = torch.abs(feat) + 0.25\n        cauchy_w = 1.0 / (1.0 + (diff / scale) ** 2)\n        return feat + torch.tanh(gate.to(dtype=feat.dtype)) * (diff * cauchy_w)\n\n    def _pool_stage(self, feat: torch.Tensor, proj: nn.Module) -> torch.Tensor:\n        if feat.ndim == 2:\n            feat = feat.unsqueeze(-1).unsqueeze(-1)\n        p = proj(feat.contiguous())\n        return torch.cat([torch.flatten(self.avg_pool(p), 1), torch.flatten(self.max_pool(p), 1)], dim=-1)\n\n    def forward(self, s2: torch.Tensor, s3: torch.Tensor, s4: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        s2_att = self._apply_bilateral_symmetry(self.coord_att2(s2), self.sym_gate2)\n        s3_att = self._apply_bilateral_symmetry(self.coord_att3(s3), self.sym_gate3)\n        z2 = self._pool_stage(s2_att, self.proj2)\n        z3 = self._pool_stage(s3_att, self.proj3)\n        z4 = self._pool_stage(s4, self.proj4)\n        z_base = self.fusion(torch.cat([z2, z3, z4], dim=-1))\n        z_fused = self.fusion_norm(z_base + 0.25 * self.fusion_grn_block(z_base))\n        return z_fused, z4\n\nclass BiometricMultiTaskModel(nn.Module):\n    def __init__(self, backbone_name: str = "convnextv2_tiny", num_age_bins: int = 101, reid_dim: int = 64, pretrained: bool = True, drop_path_rate: float = 0.15):\n        super().__init__()\n        self.backbone_name = backbone_name\n        self.num_age_bins = num_age_bins\n        self.num_fine_intervals = NUM_FINE_INTERVALS\n        self.reid_dim = reid_dim\n\n        self.photometric_stem = PhotometricRetinexStem(hidden_channels=24)\n        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=True, drop_path_rate=drop_path_rate)\n        ch = self.backbone.feature_info.channels()\n        stage_channels = (ch[-3], ch[-2], ch[-1]) if len(ch) >= 3 else (ch[0], ch[0], ch[-1])\n        self.fusion = MultiScaleBiometricFusion(in_channels=stage_channels, proj_dim=128, fused_dim=512)\n\n        # Register K x 23 partition matrix P\n        partition = torch.zeros(num_age_bins, self.num_fine_intervals, dtype=torch.float32)\n        for idx, (_, low, high) in enumerate(FINE_AGE_INTERVALS):\n            if int(low) < num_age_bins:\n                lo_c, hi_c = max(0, int(low)), min(num_age_bins - 1, int(high))\n                if hi_c >= lo_c:\n                    partition[lo_c : hi_c + 1, idx] = 1.0\n        self.register_buffer("fine_interval_partition", partition)\n        self.register_buffer("age_bins_vec", torch.arange(num_age_bins, dtype=torch.float32))\n\n        self.age_interval_head = nn.Sequential(\n            nn.Linear(512, 192), nn.SiLU(), GlobalResponseNorm1d(192), nn.Linear(192, self.num_fine_intervals)\n        )\n        self.hier_interval_scale = nn.Parameter(torch.tensor(0.35, dtype=torch.float32))\n        self.age_ldl_head = nn.Sequential(\n            nn.Linear(512, 256), nn.SiLU(), GlobalResponseNorm1d(256), nn.Linear(256, num_age_bins)\n        )\n        self.gender_head = nn.Sequential(\n            nn.Linear(256, 128), nn.SiLU(), GlobalResponseNorm1d(128), nn.Linear(128, 1)\n        )\n        self.uncertainty_head = nn.Sequential(\n            nn.Linear(512, 64), nn.SiLU(), nn.Linear(64, 1)\n        )\n        self.reid_head = nn.Sequential(\n            nn.Linear(256, reid_dim), nn.LayerNorm(reid_dim)\n        )\n\n    def forward(self, x: torch.Tensor, return_dict: bool = True):\n        x_norm = self.photometric_stem(x)\n        feats = self.backbone(x_norm)\n        s2, s3, s4 = (feats[-3], feats[-2], feats[-1]) if len(feats) >= 3 else (feats[0], feats[0], feats[-1])\n        z_fused, z4 = self.fusion(s2, s3, s4)\n\n        fine_interval_logits = self.age_interval_head(z_fused)\n        fine_interval_log_prior = F.log_softmax(fine_interval_logits, dim=-1)\n        part_mat = self.fine_interval_partition.to(dtype=fine_interval_log_prior.dtype)\n        lifted_interval_prior = torch.matmul(fine_interval_log_prior, part_mat.T)\n        scale = F.softplus(self.hier_interval_scale.to(dtype=z_fused.dtype))\n        age_logits = self.age_ldl_head(z_fused) + scale * lifted_interval_prior\n        age_probs = F.softmax(age_logits, dim=-1)\n        expected_age = torch.sum(age_probs * self.age_bins_vec.to(dtype=age_probs.dtype), dim=-1)\n\n        part_mat_probs = self.fine_interval_partition.to(dtype=age_probs.dtype)\n        part_probs = torch.matmul(age_probs, part_mat_probs)\n        if self.num_age_bins == 101:\n            head_probs = F.softmax(fine_interval_logits, dim=-1).to(dtype=age_probs.dtype)\n            bins_row = self.age_bins_vec.to(dtype=age_probs.dtype).unsqueeze(0)\n            gauss_focus = torch.exp(-0.5 * ((bins_row - expected_age.unsqueeze(-1)) / 1.6) ** 2)\n            focused_probs = age_probs * (0.20 + 0.80 * gauss_focus)\n            focused_part_probs = torch.matmul(focused_probs, part_mat_probs)\n            focus_iv = torch.matmul(gauss_focus, part_mat_probs)\n            age_interval_probs = 0.85 * focused_part_probs + 0.15 * (head_probs * focus_iv)\n        else:\n            age_interval_probs = part_probs\n        age_interval_probs = age_interval_probs / torch.clamp(age_interval_probs.sum(dim=-1, keepdim=True), min=1e-12)\n        age_interval_idx = torch.argmax(age_interval_probs, dim=-1)\n\n        gender_logit = self.gender_head(z4).squeeze(-1)\n        gender_prob = torch.sigmoid(gender_logit)\n        log_var = torch.clamp(self.uncertainty_head(z_fused).squeeze(-1), min=-6.0, max=4.0)\n        aleatoric_std = torch.exp(0.5 * log_var)\n        reid_emb = F.normalize(self.reid_head(z4), p=2, dim=-1)\n\n        if torch.jit.is_tracing() or not return_dict:\n            return age_logits, gender_logit, log_var, reid_emb\n\n        return {\n            "age_logits": age_logits,\n            "age_probs": age_probs,\n            "age_ldl": age_probs,\n            "fine_interval_logits": fine_interval_logits,\n            "age_interval_probs": age_interval_probs,\n            "age_interval_idx": age_interval_idx,\n            "expected_age": expected_age,\n            "gender_logit": gender_logit,\n            "gender_prob": gender_prob,\n            "log_variance": log_var,\n            "aleatoric_std": aleatoric_std,\n            "reid_embedding": reid_emb,\n            "z_fused": z_fused,\n            "z4": z4,\n        }\n\n    def predict_smooth(self, x: torch.Tensor, enable_flip_tta: bool = True) -> Dict[str, torch.Tensor]:\n        if not enable_flip_tta:\n            out = self.forward(x, return_dict=True)\n            out["epistemic_std"] = torch.zeros_like(out["aleatoric_std"])\n            out["total_std"] = out["aleatoric_std"]\n            return out\n        b = x.shape[0]\n        x_pair = torch.cat([x, torch.flip(x, dims=[3])], dim=0)\n        out_pair = self.forward(x_pair, return_dict=True)\n        p_orig, p_flip = out_pair["age_probs"][:b], out_pair["age_probs"][b:]\n        age_probs = 0.5 * (p_orig + p_flip)\n        age_probs = age_probs / torch.clamp(age_probs.sum(dim=-1, keepdim=True), min=1e-12)\n        expected_age = torch.sum(age_probs * self.age_bins_vec.to(dtype=age_probs.dtype), dim=-1)\n        fine_iv_logits = 0.5 * (out_pair["fine_interval_logits"][:b] + out_pair["fine_interval_logits"][b:])\n        part_mat_probs = self.fine_interval_partition.to(dtype=age_probs.dtype)\n        bins_row = self.age_bins_vec.to(dtype=age_probs.dtype).unsqueeze(0)\n        gauss_focus = torch.exp(-0.5 * ((bins_row - expected_age.unsqueeze(-1)) / 1.6) ** 2)\n        focused_probs = age_probs * (0.20 + 0.80 * gauss_focus)\n        age_interval_probs = torch.matmul(focused_probs, part_mat_probs)\n        if self.num_age_bins == 101:\n            iv_head_probs = F.softmax(fine_iv_logits, dim=-1).to(dtype=age_probs.dtype)\n            focus_iv = torch.matmul(gauss_focus, part_mat_probs)\n            age_interval_probs = 0.85 * age_interval_probs + 0.15 * (iv_head_probs * focus_iv)\n        age_interval_probs = age_interval_probs / torch.clamp(age_interval_probs.sum(dim=-1, keepdim=True), min=1e-12)\n        age_interval_idx = torch.argmax(age_interval_probs, dim=-1)\n        g_logit = 0.5 * (out_pair["gender_logit"][:b] + out_pair["gender_logit"][b:])\n        g_prob = torch.sigmoid(g_logit)\n        a_std = 0.5 * (out_pair["aleatoric_std"][:b] + out_pair["aleatoric_std"][b:])\n        e_age_diff = 0.5 * torch.abs(out_pair["expected_age"][:b] - out_pair["expected_age"][b:])\n        w_half = x.shape[3] // 2\n        if w_half > 0:\n            left_h = x[:, :, :, :w_half]\n            right_h = torch.flip(x[:, :, :, -w_half:], dims=[3])\n            hemiface_var_gap = torch.abs(torch.var(left_h, dim=(1, 2, 3), unbiased=False) - torch.var(right_h, dim=(1, 2, 3), unbiased=False))\n            bilateral_pen = 0.50 * hemiface_var_gap\n        else:\n            bilateral_pen = torch.zeros_like(a_std)\n        e_std = torch.sqrt(e_age_diff ** 2 + bilateral_pen ** 2)\n        g_dis = 0.05 * ((out_pair["gender_logit"][:b] - out_pair["gender_logit"][b:]) ** 2)\n        total_std = torch.sqrt(0.5 * (out_pair["aleatoric_std"][:b] ** 2 + out_pair["aleatoric_std"][b:] ** 2) + e_std ** 2 + g_dis + 1e-8)\n        reid_emb = F.normalize(0.5 * (out_pair["reid_embedding"][:b] + out_pair["reid_embedding"][b:]), p=2, dim=-1)\n        return {\n            "age_logits": 0.5 * (out_pair["age_logits"][:b] + out_pair["age_logits"][b:]),\n            "age_probs": age_probs,\n            "age_ldl": age_probs,\n            "fine_interval_logits": fine_iv_logits,\n            "age_interval_probs": age_interval_probs,\n            "age_interval_idx": age_interval_idx,\n            "expected_age": expected_age,\n            "gender_logit": g_logit,\n            "gender_prob": g_prob,\n            "log_variance": 0.5 * (out_pair["log_variance"][:b] + out_pair["log_variance"][b:]),\n            "aleatoric_std": a_std,\n            "epistemic_std": e_std,\n            "total_std": total_std,\n            "reid_embedding": reid_emb,\n        }\n\n# ---------------------------------------------------------------------------\n# 6. Multi-Task Biometric Losses & Photometric Consistency Objective\n# ---------------------------------------------------------------------------\ndef interval_censored_ordinal_loss(pred_probs: torch.Tensor, age_min: torch.Tensor, age_max: torch.Tensor, mask=None, sample_weights=None, eps: float = 1e-8) -> torch.Tensor:\n    B, K = pred_probs.shape\n    p = pred_probs.float() / (torch.sum(pred_probs.float(), dim=-1, keepdim=True) + 1e-12)\n    bins = torch.arange(K, device=p.device, dtype=torch.float32).unsqueeze(0)\n    a_low = torch.clamp(torch.minimum(age_min.float(), age_max.float()), 0.0, float(K - 1)).unsqueeze(1)\n    a_high = torch.clamp(torch.maximum(age_min.float(), age_max.float()), 0.0, float(K - 1)).unsqueeze(1)\n    in_interval = ((bins >= (a_low - 0.5)) & (bins <= (a_high + 0.5))).float()\n    mass_in_interval = torch.sum(p * in_interval, dim=-1)\n    nll = -torch.log(torch.clamp(mass_in_interval, min=eps, max=1.0))\n    cdf_p = torch.cumsum(p, dim=-1)\n    below_mask = (bins < (a_low - 0.5)).float()\n    above_mask = (bins >= (a_high + 0.5)).float()\n    boundary_penalty = torch.sum((cdf_p ** 2) * below_mask + ((1.0 - cdf_p) ** 2) * above_mask, dim=-1)\n    per_sample = nll + 0.25 * boundary_penalty\n    m = mask.float().view(-1) if mask is not None else torch.ones(B, device=p.device, dtype=torch.float32)\n    valid_sum = torch.sum(m)\n    if valid_sum <= 1e-8: return torch.sum(per_sample * 0.0)\n    w = m * sample_weights.float().view(-1) if sample_weights is not None else m\n    return torch.sum(per_sample * w) / (valid_sum + 1e-8)\n\ndef fine_interval_ldl_loss(pred_interval_probs: torch.Tensor, target_interval_probs: torch.Tensor, mask=None, sample_weights=None, eps: float = 1e-8) -> torch.Tensor:\n    p = pred_interval_probs.float() / (torch.sum(pred_interval_probs.float(), dim=-1, keepdim=True) + 1e-12)\n    q = target_interval_probs.float() / (torch.sum(target_interval_probs.float(), dim=-1, keepdim=True) + 1e-12)\n    cdf_diff = torch.cumsum(p, dim=-1) - torch.cumsum(q, dim=-1)\n    cramer_23 = torch.sum(cdf_diff ** 2, dim=-1)\n    p_safe = torch.clamp(p, min=eps)\n    q_safe = torch.clamp(q, min=eps)\n    kl_23 = torch.clamp(torch.sum(q * (torch.log(q_safe) - torch.log(p_safe)), dim=-1), min=0.0)\n    per_sample = cramer_23 + 0.5 * kl_23\n    m = mask.float().view(-1) if mask is not None else torch.ones(p.shape[0], device=p.device, dtype=torch.float32)\n    valid_sum = torch.sum(m)\n    if valid_sum <= 1e-8: return torch.sum(per_sample * 0.0)\n    w = m * sample_weights.float().view(-1) if sample_weights is not None else m\n    return torch.sum(per_sample * w) / (valid_sum + 1e-8)\n\ndef ldl_tikhonov_smoothness_loss(pred_probs: torch.Tensor, mask=None, sample_weights=None) -> torch.Tensor:\n    p = pred_probs.float() / (torch.sum(pred_probs.float(), dim=-1, keepdim=True) + 1e-12)\n    laplacian = p[:, 2:] - 2.0 * p[:, 1:-1] + p[:, :-2]\n    per_sample = torch.sum(laplacian ** 2, dim=-1) * 10.0\n    m = mask.float().view(-1) if mask is not None else torch.ones(p.shape[0], device=p.device, dtype=torch.float32)\n    valid_sum = torch.sum(m)\n    if valid_sum <= 1e-8: return torch.sum(per_sample * 0.0)\n    w = m * sample_weights.float().view(-1) if sample_weights is not None else m\n    return torch.sum(per_sample * w) / (valid_sum + 1e-8)\n\ndef photometric_consistency_loss(preds_clean: Dict[str, torch.Tensor], preds_corrupted: Dict[str, torch.Tensor], age_mask=None, gender_mask=None, eps: float = 1e-8) -> torch.Tensor:\n    p1 = preds_clean["age_probs"].detach().float()\n    p1 = p1 / (torch.sum(p1, dim=-1, keepdim=True) + 1e-12)\n    p2 = preds_corrupted["age_probs"].float()\n    p2 = p2 / (torch.sum(p2, dim=-1, keepdim=True) + 1e-12)\n    m_dist = 0.5 * (p1 + p2)\n    p1_s, p2_s = torch.clamp(p1, min=eps), torch.clamp(p2, min=eps)\n    log_m = torch.log(torch.clamp(m_dist, min=eps))\n    kl_1 = torch.sum(p1 * (torch.log(p1_s) - log_m), dim=-1)\n    kl_2 = torch.sum(p2 * (torch.log(p2_s) - log_m), dim=-1)\n    js_div = torch.clamp(0.5 * (kl_1 + kl_2), min=0.0)\n    cramer = torch.sum((torch.cumsum(p1, dim=-1) - torch.cumsum(p2, dim=-1)) ** 2, dim=-1)\n    age_cons = js_div + 0.5 * cramer\n    if age_mask is not None:\n        am = age_mask.float().view(-1)\n        l_age_cons = torch.sum(age_cons * am) / torch.clamp(torch.sum(am), min=1.0)\n    else:\n        l_age_cons = torch.mean(age_cons)\n    g1 = torch.sigmoid(preds_clean["gender_logit"].detach().float())\n    g2 = torch.sigmoid(preds_corrupted["gender_logit"].float())\n    g_cons = (g1 - g2) ** 2\n    if gender_mask is not None:\n        gm = gender_mask.float().view(-1)\n        l_gender_cons = torch.sum(g_cons * gm) / torch.clamp(torch.sum(gm), min=1.0)\n    else:\n        l_gender_cons = torch.mean(g_cons)\n    r1 = F.normalize(preds_clean["reid_embedding"].detach().float(), p=2, dim=-1)\n    r2 = F.normalize(preds_corrupted["reid_embedding"].float(), p=2, dim=-1)\n    l_reid_cons = torch.mean(1.0 - torch.sum(r1 * r2, dim=-1))\n    return l_age_cons + 0.5 * l_gender_cons + 0.25 * l_reid_cons\n\nclass CoupledHeteroscedasticLaplaceLoss(nn.Module):\n    def __init__(self, delta: float = 1.0, min_log_var: float = -6.0, max_log_var: float = 4.0):\n        super().__init__()\n        self.delta, self.min_log_var, self.max_log_var = delta, min_log_var, max_log_var\n\n    def forward(self, expected_age, target_age, log_variance, sample_weights=None, age_mask=None):\n        mu = expected_age.float().view(-1)\n        target = target_age.float().view(-1)\n        s = torch.clamp(log_variance.float().view(-1), self.min_log_var, self.max_log_var)\n        huber = F.huber_loss(mu, target, delta=self.delta, reduction="none")\n        loss = torch.exp(-s) * huber + 0.5 * s\n        m = age_mask.float().view(-1) if age_mask is not None else torch.ones_like(loss)\n        valid_m = torch.sum(m)\n        if valid_m <= 1e-8: return torch.sum(loss * 0.0)\n        w = m * sample_weights.float().view(-1) if sample_weights is not None else m\n        return torch.sum(loss * w) / (valid_m + 1e-8)\n\nclass SupervisedContrastiveLoss(nn.Module):\n    def __init__(self, temperature: float = 0.07):\n        super().__init__()\n        self.temperature = float(temperature)\n\n    def forward(self, embeddings: torch.Tensor, labels: Optional[torch.Tensor] = None, sample_weights: Optional[torch.Tensor] = None) -> torch.Tensor:\n        device = embeddings.device\n        B = embeddings.shape[0]\n        if B <= 1 or labels is None: return torch.sum(embeddings * 0.0)\n        labels_1d = labels.contiguous().view(-1)\n        if sample_weights is not None:\n            alpha = torch.clamp(sample_weights.float().view(-1), min=0.0)\n            keep = alpha > 0.0\n            if int(keep.sum().item()) <= 1: return torch.sum(embeddings * 0.0)\n            embeddings, labels_1d, alpha = embeddings[keep], labels_1d[keep], alpha[keep]\n            B = embeddings.shape[0]\n        else:\n            alpha = torch.ones(B, device=device, dtype=torch.float32)\n        norm_emb = F.normalize(embeddings.float(), p=2, dim=-1)\n        labels_col = labels_1d.view(-1, 1)\n        valid_labels = (labels_col >= 0).float()\n        pos_mask = torch.eq(labels_col, labels_col.T).float() * (valid_labels @ valid_labels.T)\n        diag_mask = torch.scatter(torch.ones_like(pos_mask), 1, torch.arange(B, device=device).view(-1, 1), 0.0)\n        pos_mask = pos_mask * diag_mask\n        sim = torch.div(torch.matmul(norm_emb, norm_emb.T), self.temperature)\n        sim_off_diag = sim - (1.0 - diag_mask) * 1e9\n        sim_max, _ = torch.max(sim_off_diag, dim=1, keepdim=True)\n        logits = sim - sim_max.detach()\n        denom = torch.log((torch.exp(logits) * diag_mask).sum(1, keepdim=True) + 1e-12)\n        log_prob = logits - denom\n        pos_counts = pos_mask.sum(1)\n        has_positives = pos_counts > 0\n        if not has_positives.any(): return torch.sum(norm_emb * 0.0)\n        mean_log_prob_pos = (pos_mask * log_prob).sum(1)[has_positives].div(pos_counts[has_positives] + 1e-12)\n        return -torch.mean(mean_log_prob_pos * alpha[has_positives])\n\ndef refine_interval_ldl_with_teacher(\n    box_ldl_101: torch.Tensor,\n    teacher_age_probs: torch.Tensor,\n    age_min: torch.Tensor,\n    age_max: torch.Tensor,\n    blend_alpha: float = 0.65,\n    margin: float = 1.5,\n) -> Tuple[torch.Tensor, torch.Tensor]:\n    B, K = box_ldl_101.shape\n    device_t = box_ldl_101.device\n    bins = torch.arange(K, device=device_t, dtype=torch.float32).unsqueeze(0)\n    a_low = torch.clamp(torch.minimum(age_min.float(), age_max.float()).view(-1, 1), 0.0, float(K - 1))\n    a_high = torch.clamp(torch.maximum(age_min.float(), age_max.float()).view(-1, 1), 0.0, float(K - 1))\n    span = (a_high - a_low).squeeze(-1)\n\n    p_box = box_ldl_101.float() / torch.clamp(box_ldl_101.float().sum(dim=-1, keepdim=True), min=1e-12)\n    p_teach = teacher_age_probs.detach().float()\n    p_teach = p_teach / torch.clamp(p_teach.sum(dim=-1, keepdim=True), min=1e-12)\n\n    dist_below = torch.clamp(a_low - bins, min=0.0)\n    dist_above = torch.clamp(bins - a_high, min=0.0)\n    dist_out = dist_below + dist_above\n    window = torch.exp(-0.5 * (dist_out / max(margin, 1e-3)) ** 2)\n\n    p_proj = p_teach * window\n    proj_mass = p_proj.sum(dim=-1, keepdim=True)\n    p_proj = torch.where(proj_mass > 1e-6, p_proj / torch.clamp(proj_mass, min=1e-12), p_box)\n\n    alpha = float(np.clip(blend_alpha, 0.0, 1.0))\n    is_interval = (span >= 2.0).float().unsqueeze(-1)\n    eff_alpha = alpha * is_interval\n    refined_101 = (1.0 - eff_alpha) * p_box + eff_alpha * p_proj\n    refined_101 = refined_101 / torch.clamp(refined_101.sum(dim=-1, keepdim=True), min=1e-12)\n\n    refined_23 = torch.zeros((B, NUM_FINE_INTERVALS), device=device_t, dtype=torch.float32)\n    for idx, (_, low, high) in enumerate(FINE_AGE_INTERVALS):\n        lo_c, hi_c = max(0, int(low)), min(K - 1, int(high))\n        if hi_c >= lo_c:\n            refined_23[:, idx] = refined_101[:, lo_c : hi_c + 1].sum(dim=-1)\n    refined_23 = refined_23 / torch.clamp(refined_23.sum(dim=-1, keepdim=True), min=1e-12)\n    return refined_101, refined_23\n\n\nclass BiometricDistillationLoss(nn.Module):\n    def __init__(\n        self,\n        temperature: float = 2.0,\n        w_age_kl: float = 0.85,\n        w_age_cramer: float = 0.65,\n        w_interval_kl: float = 0.65,\n        w_gender: float = 0.90,\n        w_reid: float = 0.45,\n        gender_conf_thresh: float = 0.80,\n    ):\n        super().__init__()\n        self.temperature = float(max(temperature, 0.1))\n        self.w_age_kl = float(w_age_kl)\n        self.w_age_cramer = float(w_age_cramer)\n        self.w_interval_kl = float(w_interval_kl)\n        self.w_gender = float(w_gender)\n        self.w_reid = float(w_reid)\n        self.gender_conf_thresh = float(np.clip(gender_conf_thresh, 0.5, 0.99))\n\n    def forward(\n        self,\n        student_preds: Dict[str, torch.Tensor],\n        teacher_targets: Dict[str, torch.Tensor],\n        targets: Optional[Dict[str, torch.Tensor]] = None,\n    ) -> Tuple[torch.Tensor, Dict[str, float]]:\n        T = self.temperature\n        s_age_logits = student_preds["age_logits"].float()\n        t_age_logits = teacher_targets["teacher_age_logits"].detach().float()\n\n        B = s_age_logits.shape[0]\n        device_t = s_age_logits.device\n        if targets is not None and "has_valid_age" in targets:\n            age_mask = targets["has_valid_age"].float().view(-1)\n        else:\n            age_mask = torch.ones(B, device=device_t, dtype=torch.float32)\n        valid_age_cnt = torch.clamp(age_mask.sum(), min=1.0)\n\n        log_p_s = F.log_softmax(s_age_logits / T, dim=-1)\n        p_t = F.softmax(t_age_logits / T, dim=-1)\n        kl_101_per = torch.sum(F.kl_div(log_p_s, p_t, reduction="none"), dim=-1) * (T * T)\n        l_age_kl = torch.sum(kl_101_per * age_mask) / valid_age_cnt\n\n        prob_s = student_preds["age_probs"].float()\n        prob_s = prob_s / torch.clamp(prob_s.sum(dim=-1, keepdim=True), min=1e-12)\n        prob_t = teacher_targets["teacher_age_probs"].detach().float()\n        prob_t = prob_t / torch.clamp(prob_t.sum(dim=-1, keepdim=True), min=1e-12)\n        cramer_per = torch.sum((torch.cumsum(prob_s, dim=-1) - torch.cumsum(prob_t, dim=-1)) ** 2, dim=-1)\n        l_age_cramer = torch.sum(cramer_per * age_mask) / valid_age_cnt\n\n        s_iv_logits = student_preds["fine_interval_logits"].float()\n        t_iv_logits = teacher_targets["teacher_fine_interval_logits"].detach().float()\n        log_iv_s = F.log_softmax(s_iv_logits / T, dim=-1)\n        iv_t = F.softmax(t_iv_logits / T, dim=-1)\n        kl_23_per = torch.sum(F.kl_div(log_iv_s, iv_t, reduction="none"), dim=-1) * (T * T)\n        l_iv_kl = torch.sum(kl_23_per * age_mask) / valid_age_cnt\n\n        s_g_logit = student_preds["gender_logit"].float().view(-1)\n        t_g_logit = teacher_targets["teacher_gender_logit"].detach().float().view(-1)\n        t_g_prob = torch.sigmoid(t_g_logit)\n        t_g_conf = torch.maximum(t_g_prob, 1.0 - t_g_prob)\n        if targets is not None and "gender_mask" in targets:\n            g_mask = targets["gender_mask"].float().view(-1)\n            pseudo_mask = ((g_mask < 0.5) & (t_g_conf >= self.gender_conf_thresh)).float()\n            eff_g_mask = torch.clamp(g_mask + 0.75 * pseudo_mask, min=0.0, max=1.0)\n        else:\n            eff_g_mask = torch.ones_like(s_g_logit)\n        if targets is not None and "dimorphism_weight" in targets:\n            eff_g_mask = eff_g_mask * targets["dimorphism_weight"].float().view(-1)\n\n        bce_kd = F.binary_cross_entropy_with_logits(s_g_logit, t_g_prob, reduction="none")\n        valid_g = torch.clamp(eff_g_mask.sum(), min=1.0)\n        l_gender_kd = torch.sum(bce_kd * eff_g_mask) / valid_g\n\n        s_emb = F.normalize(student_preds["reid_embedding"].float(), p=2, dim=-1)\n        t_emb = F.normalize(teacher_targets["teacher_reid_embedding"].detach().float(), p=2, dim=-1)\n        cos_loss = torch.mean(1.0 - torch.sum(s_emb * t_emb, dim=-1))\n        if B > 1:\n            sim_s = torch.matmul(s_emb, s_emb.T)\n            sim_t = torch.matmul(t_emb, t_emb.T)\n            rel_loss = F.smooth_l1_loss(sim_s, sim_t, beta=0.1)\n            l_reid_kd = cos_loss + 0.5 * rel_loss\n        else:\n            l_reid_kd = cos_loss\n\n        total_kd = (\n            self.w_age_kl * l_age_kl\n            + self.w_age_cramer * l_age_cramer\n            + self.w_interval_kl * l_iv_kl\n            + self.w_gender * l_gender_kd\n            + self.w_reid * l_reid_kd\n        )\n        return total_kd, {\n            "kd_age_kl": float(l_age_kl.item()),\n            "kd_age_cramer": float(l_age_cramer.item()),\n            "kd_interval_kl": float(l_iv_kl.item()),\n            "kd_gender": float(l_gender_kd.item()),\n            "kd_reid": float(l_reid_kd.item()),\n        }\n\n\nclass UnifiedBiometricLoss(nn.Module):\n    def __init__(self, w_cramer=1.0, w_kl=0.5, w_mae=0.25, w_interval=0.50, w_fine_23=0.80, w_smoothness=0.15, w_gender=1.2, w_unc=0.1, w_reid=0.15, reid_temp=0.07):\n        super().__init__()\n        self.w_cramer, self.w_kl, self.w_mae = w_cramer, w_kl, w_mae\n        self.w_interval, self.w_fine_23, self.w_smoothness = w_interval, w_fine_23, w_smoothness\n        self.w_gender, self.w_unc, self.w_reid = w_gender, w_unc, w_reid\n        self.supcon = SupervisedContrastiveLoss(temperature=reid_temp)\n        self.coupled_laplace = CoupledHeteroscedasticLaplaceLoss()\n\n    def forward(self, preds: Dict[str, torch.Tensor], targets: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, Dict[str, float]]:\n        age_mask = targets.get("has_valid_age", torch.ones_like(targets["age_continuous"])).float().view(-1)\n        valid_age_cnt = torch.clamp(torch.sum(age_mask), min=1e-8)\n        alpha_vec = torch.clamp(targets.get("alpha_sample_weight", torch.ones_like(age_mask)).float().view(-1), min=0.0)\n        eff_age_w = age_mask * alpha_vec\n\n        p = preds["age_probs"].float() / (preds["age_probs"].float().sum(-1, keepdim=True) + 1e-12)\n        q = targets["ldl_101"].float() / (targets["ldl_101"].float().sum(-1, keepdim=True) + 1e-12)\n        cramer_per = torch.sum((torch.cumsum(p, -1) - torch.cumsum(q, -1)) ** 2, -1)\n        l_cramer = torch.sum(cramer_per * eff_age_w) / valid_age_cnt\n\n        log_p = F.log_softmax(preds["age_logits"].float(), dim=-1)\n        kl_per = torch.sum(F.kl_div(log_p, q, reduction="none"), dim=-1)\n        l_kl = torch.sum(kl_per * eff_age_w) / valid_age_cnt\n\n        mae_per = torch.abs(preds["expected_age"].float() - targets["age_continuous"].float())\n        l_mae = torch.sum(mae_per * eff_age_w) / valid_age_cnt\n\n        l_interval = interval_censored_ordinal_loss(p, targets["age_min"], targets["age_max"], mask=age_mask, sample_weights=alpha_vec)\n\n        if "fine_interval_logits" in preds:\n            p_23 = F.softmax(preds["fine_interval_logits"].float(), dim=-1)\n            log_p_23 = F.log_softmax(preds["fine_interval_logits"].float(), dim=-1)\n            q_23 = targets["ldl_fine_23"].float() / (targets["ldl_fine_23"].float().sum(-1, keepdim=True) + 1e-12)\n            soft_ce_23 = -torch.sum(q_23 * log_p_23, dim=-1)\n            l_soft_ce_23 = torch.sum(soft_ce_23 * eff_age_w) / valid_age_cnt\n        else:\n            p_23 = preds["age_interval_probs"].float()\n            l_soft_ce_23 = torch.sum(p * 0.0)\n        l_fine_23 = fine_interval_ldl_loss(p_23, targets["ldl_fine_23"], mask=age_mask, sample_weights=alpha_vec)\n        if "age_interval_probs" in preds and "fine_interval_logits" in preds:\n            l_fine_23 = 0.45 * l_fine_23 + 0.35 * fine_interval_ldl_loss(preds["age_interval_probs"], targets["ldl_fine_23"], mask=age_mask, sample_weights=alpha_vec) + 0.20 * l_soft_ce_23\n\n        l_smooth = ldl_tikhonov_smoothness_loss(p, mask=age_mask, sample_weights=alpha_vec)\n\n        bce = F.binary_cross_entropy_with_logits(preds["gender_logit"].float(), targets["gender"].float(), reduction="none")\n        g_mask = targets["gender_mask"].float().view(-1)\n        dim_w = targets["dimorphism_weight"].float().view(-1)\n        valid_g_cnt = torch.sum(g_mask)\n        l_gender = torch.sum(bce * g_mask * dim_w * alpha_vec) / (valid_g_cnt + 1e-8) if valid_g_cnt > 0 else torch.sum(bce * 0.0)\n\n        l_unc = self.coupled_laplace(preds["expected_age"], targets["age_continuous"], preds["log_variance"], sample_weights=alpha_vec, age_mask=age_mask)\n\n        reid_emb = preds.get("reid_embedding")\n        id_labels = targets.get("identity")\n        if reid_emb is not None and id_labels is not None and (id_labels >= 0).any():\n            l_reid = self.supcon(reid_emb, id_labels, sample_weights=alpha_vec)\n        else:\n            l_reid = torch.sum(p * 0.0)\n\n        total = (\n            self.w_cramer * l_cramer\n            + self.w_kl * l_kl\n            + self.w_mae * l_mae\n            + self.w_interval * l_interval\n            + self.w_fine_23 * l_fine_23\n            + self.w_smoothness * l_smooth\n            + self.w_gender * l_gender\n            + self.w_unc * l_unc\n            + self.w_reid * l_reid\n        )\n        return total, {\n            "cramer": float(l_cramer.item()),\n            "kl": float(l_kl.item()),\n            "mae": float(l_mae.item()),\n            "interval": float(l_interval.item()),\n            "fine_23": float(l_fine_23.item()),\n            "smooth": float(l_smooth.item()),\n            "gender": float(l_gender.item()),\n            "unc": float(l_unc.item()),\n            "reid": float(l_reid.item()),\n        }\n\n# ---------------------------------------------------------------------------\n# 7. Model Exponential Moving Average (EMA) & TargetLocker Temporal Smoothing\n# ---------------------------------------------------------------------------\nclass ModelEMA:\n    def __init__(self, model: nn.Module, decay: float = 0.992):\n        self.ema = copy.deepcopy(model).eval()\n        for p in self.ema.parameters():\n            p.requires_grad_(False)\n        self.decay = float(decay)\n\n    @torch.no_grad()\n    def update(self, model: nn.Module):\n        msd = model.state_dict()\n        for k, v in self.ema.state_dict().items():\n            if k in msd:\n                src = msd[k]\n                if v.dtype.is_floating_point:\n                    v.mul_(self.decay).add_(src.detach(), alpha=1.0 - self.decay)\n                else:\n                    v.copy_(src)\n\nclass TemporalTargetLocker:\n    """Huber-Kalman Age Filter + 23-Bracket Hysteresis Deadband + Bounded Leaky Gender Accumulator."""\n    def __init__(self, initial_age: float, initial_std: float, initial_logit: float, q: float = 0.04, huber_delta: float = 5.0, hysteresis_margin: float = 0.35):\n        self.age = float(initial_age)\n        self.var = float(initial_std ** 2)\n        self.q = float(q)\n        self.huber_delta = float(huber_delta)\n        self.hysteresis_margin = float(hysteresis_margin)\n        self.log_odds = float(np.clip(initial_logit, -6.0, 6.0))\n        _, self.interval_idx, _, _ = get_fine_interval_for_age(self.age)\n\n    def update(self, measured_age: float, total_std: float, gender_logit: float) -> Tuple[float, int, str]:\n        p_pred = self.var + self.q\n        r = max(1e-4, float(total_std) ** 2)\n        innov = float(measured_age) - self.age\n        thresh = self.huber_delta * math.sqrt(p_pred + r)\n        if abs(innov) > thresh:\n            r = r * ((abs(innov) / max(thresh, 1e-6)) ** 2)\n        k = p_pred / (p_pred + r)\n        self.age = self.age + k * innov\n        self.var = (1.0 - k) * p_pred\n        self.log_odds = float(np.clip(0.95 * self.log_odds + float(gender_logit), -6.0, 6.0))\n        _, cand_idx, _, _ = get_fine_interval_for_age(self.age)\n        if cand_idx != self.interval_idx:\n            _, prev_low, prev_high = FINE_AGE_INTERVALS[self.interval_idx]\n            if not ((prev_low - 0.5 - self.hysteresis_margin) <= self.age <= (prev_high + 0.5 + self.hysteresis_margin)):\n                self.interval_idx = cand_idx\n        return self.age, self.interval_idx, ("Female" if self.log_odds >= 0.0 else "Male")\n\n# ---------------------------------------------------------------------------\n# 8. Comprehensive Evaluation Helper (Clean + 6 Extreme Noise Regimes + Fairness)\n# ---------------------------------------------------------------------------\ndef compute_binary_roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:\n    pos = y_true == 1\n    neg = y_true == 0\n    n_pos, n_neg = int(np.sum(pos)), int(np.sum(neg))\n    if n_pos == 0 or n_neg == 0:\n        return 0.5\n    order = np.argsort(y_score)\n    ranks = np.empty_like(order, dtype=np.float64)\n    ranks[order] = np.arange(1, len(y_score) + 1, dtype=np.float64)\n    auc = (np.sum(ranks[pos]) - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)\n    return float(auc)\n\n@torch.no_grad()\ndef evaluate_dataset_regime(\n    model: BiometricMultiTaskModel,\n    dataset: FaceGoldenDataset,\n    regime: str = "clean",\n    enable_flip_tta: bool = True,\n    batch_size: int = 64,\n    clean_ref_preds: Optional[Dict[str, np.ndarray]] = None,\n) -> Tuple[Dict[str, Any], Dict[str, np.ndarray]]:\n    model.eval()\n    N = len(dataset)\n    pred_ages, true_ages, valid_ages = [], [], []\n    pred_iv_probs, true_iv_idx = [], []\n    pred_g_probs, true_g, valid_g, dim_weights = [], [], [], []\n    pred_total_stds, pred_aleatoric_stds, pred_epistemic_stds = [], [], []\n    laplacian_energies, entropies = [], []\n    pred_ldl_all = []\n    ita_groups, sources = [], []\n\n    for start in range(0, N, batch_size):\n        end = min(N, start + batch_size)\n        batch_imgs = []\n        for idx in range(start, end):\n            raw_rgb = dataset.images_rgb[idx]\n            corrupt_rgb = ExtremeNoiseStressBenchmark.apply_regime_uint8(raw_rgb, regime=regime, seed=1000 + idx)\n            t_img = torch.from_numpy(corrupt_rgb.transpose(2, 0, 1)).float() / 127.5 - 1.0\n            batch_imgs.append(t_img)\n            ct = dataset.cached_targets[idx]\n            true_ages.append(float(ct["age_continuous"].item()))\n            valid_ages.append(bool(ct["has_valid_age"].item() > 0.5))\n            true_iv_idx.append(int(ct["age_interval_idx"].item()))\n            true_g.append(int(ct["gender"].item() > 0.5))\n            valid_g.append(bool(ct["gender_mask"].item() > 0.5))\n            dim_weights.append(float(ct["dimorphism_weight"].item()))\n            ita_groups.append(str(ct["ita_group"]))\n            sources.append(str(ct["dataset_source"]))\n\n        x_tensor = torch.stack(batch_imgs, dim=0).to(device)\n        out = model.predict_smooth(x_tensor, enable_flip_tta=enable_flip_tta)\n\n        p_ldl = out["age_probs"].cpu().numpy()\n        p_iv = out["age_interval_probs"].cpu().numpy()\n        p_age = out["expected_age"].cpu().numpy()\n        p_g = out["gender_prob"].cpu().numpy()\n        t_std = out["total_std"].cpu().numpy()\n        a_std = out["aleatoric_std"].cpu().numpy()\n        e_std = out["epistemic_std"].cpu().numpy()\n\n        pred_ldl_all.append(p_ldl)\n        pred_iv_probs.append(p_iv)\n        pred_ages.extend(p_age.tolist())\n        pred_g_probs.extend(p_g.tolist())\n        pred_total_stds.extend(t_std.tolist())\n        pred_aleatoric_stds.extend(a_std.tolist())\n        pred_epistemic_stds.extend(e_std.tolist())\n\n        lap = p_ldl[:, 2:] - 2.0 * p_ldl[:, 1:-1] + p_ldl[:, :-2]\n        laplacian_energies.extend(np.sum(lap ** 2, axis=1).tolist())\n        p_safe = np.clip(p_ldl, 1e-12, 1.0)\n        entropies.extend((-np.sum(p_safe * np.log(p_safe), axis=1)).tolist())\n\n    pred_ldl_arr = np.concatenate(pred_ldl_all, axis=0)\n    pred_iv_arr = np.concatenate(pred_iv_probs, axis=0)\n    pred_ages_arr = np.array(pred_ages, dtype=np.float64)\n    true_ages_arr = np.array(true_ages, dtype=np.float64)\n    valid_ages_arr = np.array(valid_ages, dtype=bool)\n    true_iv_arr = np.array(true_iv_idx, dtype=np.int64)\n    pred_g_arr = np.array(pred_g_probs, dtype=np.float64)\n    true_g_arr = np.array(true_g, dtype=np.int64)\n    valid_g_arr = np.array(valid_g, dtype=bool)\n    dim_w_arr = np.array(dim_weights, dtype=np.float64)\n    total_std_arr = np.array(pred_total_stds, dtype=np.float64)\n\n    # Age & 23-Interval Metrics\n    va_errors = np.abs(pred_ages_arr[valid_ages_arr] - true_ages_arr[valid_ages_arr])\n    age_mae = float(np.mean(va_errors))\n    age_rmse = float(np.sqrt(np.mean(va_errors ** 2)))\n    cs_3 = float(np.mean(va_errors <= 3.0))\n    cs_5 = float(np.mean(va_errors <= 5.0))\n    cs_8 = float(np.mean(va_errors <= 8.0))\n\n    pred_iv_top1 = np.argmax(pred_iv_arr[valid_ages_arr], axis=1)\n    true_iv_va = true_iv_arr[valid_ages_arr]\n    iv_dist_err = np.abs(pred_iv_top1 - true_iv_va)\n    iv_exact_acc = float(np.mean(iv_dist_err == 0))\n    iv_within_1_acc = float(np.mean(iv_dist_err <= 1))\n    iv_within_2_acc = float(np.mean(iv_dist_err <= 2))\n    iv_mean_bracket_error = float(np.mean(iv_dist_err))\n\n    # Top-2 and Top-3 ranked probability hits on 23 intervals\n    top3_indices = np.argsort(-pred_iv_arr[valid_ages_arr], axis=1)[:, :3]\n    iv_top2_hit = float(np.mean([true_iv_va[i] in top3_indices[i, :2] for i in range(len(true_iv_va))]))\n    iv_top3_hit = float(np.mean([true_iv_va[i] in top3_indices[i, :3] for i in range(len(true_iv_va))]))\n\n    # 10-Cohort Biological Age Accuracy\n    pred_cohorts = np.array([get_cohort_for_age(a)[1] for a in pred_ages_arr[valid_ages_arr]])\n    true_cohorts = np.array([get_cohort_for_age(a)[1] for a in true_ages_arr[valid_ages_arr]])\n    cohort_exact_acc = float(np.mean(pred_cohorts == true_cohorts))\n    cohort_within_1_acc = float(np.mean(np.abs(pred_cohorts - true_cohorts) <= 1))\n\n    # Gender Metrics\n    vg_scores = pred_g_arr[valid_g_arr]\n    vg_true = true_g_arr[valid_g_arr]\n    vg_pred = (vg_scores >= 0.5).astype(np.int64)\n    gender_acc = float(np.mean(vg_pred == vg_true)) if len(vg_true) > 0 else 0.0\n    tp = int(np.sum((vg_pred == 1) & (vg_true == 1)))\n    fp = int(np.sum((vg_pred == 1) & (vg_true == 0)))\n    fn = int(np.sum((vg_pred == 0) & (vg_true == 1)))\n    prec = float(tp / max(1, tp + fp))\n    rec = float(tp / max(1, tp + fn))\n    f1 = float(2.0 * prec * rec / max(1e-8, prec + rec))\n    roc_auc = compute_binary_roc_auc(vg_true, vg_scores) if len(vg_true) > 0 else 0.5\n\n    adult_mask = valid_g_arr & (dim_w_arr >= 0.70)\n    adult_gender_acc = float(np.mean((pred_g_arr[adult_mask] >= 0.5).astype(np.int64) == true_g_arr[adult_mask])) if np.any(adult_mask) else gender_acc\n\n    # Uncertainty & Smoothness Metrics\n    unc_err_corr = float(np.corrcoef(total_std_arr[valid_ages_arr], va_errors)[0, 1]) if len(va_errors) > 2 else 0.0\n    if math.isnan(unc_err_corr): unc_err_corr = 0.0\n\n    metrics: Dict[str, Any] = {\n        "regime": regime,\n        "num_samples": int(N),\n        "age_mae": round(age_mae, 4),\n        "age_rmse": round(age_rmse, 4),\n        "cs_at_3": round(cs_3, 4),\n        "cs_at_5": round(cs_5, 4),\n        "cs_at_8": round(cs_8, 4),\n        "fine_23_interval_exact_acc": round(iv_exact_acc, 4),\n        "fine_23_interval_within_1_bracket_acc": round(iv_within_1_acc, 4),\n        "fine_23_interval_within_2_brackets_acc": round(iv_within_2_acc, 4),\n        "fine_23_interval_top2_prob_acc": round(iv_top2_hit, 4),\n        "fine_23_interval_top3_prob_acc": round(iv_top3_hit, 4),\n        "fine_23_mean_bracket_distance": round(iv_mean_bracket_error, 4),\n        "biological_10_cohort_exact_acc": round(cohort_exact_acc, 4),\n        "biological_10_cohort_within_1_acc": round(cohort_within_1_acc, 4),\n        "gender_accuracy": round(gender_acc, 4),\n        "gender_adult_dimorphic_accuracy": round(adult_gender_acc, 4),\n        "gender_precision": round(prec, 4),\n        "gender_recall": round(rec, 4),\n        "gender_f1": round(f1, 4),\n        "gender_roc_auc": round(roc_auc, 4),\n        "mean_total_std": round(float(np.mean(total_std_arr)), 4),\n        "mean_aleatoric_std": round(float(np.mean(pred_aleatoric_stds)), 4),\n        "mean_epistemic_std": round(float(np.mean(pred_epistemic_stds)), 4),\n        "uncertainty_error_correlation": round(unc_err_corr, 4),\n        "mean_ldl_laplacian_roughness": round(float(np.mean(laplacian_energies)), 6),\n        "mean_ldl_entropy": round(float(np.mean(entropies)), 4),\n    }\n\n    # Cross-regime prediction stability relative to clean predictions\n    if clean_ref_preds is not None:\n        age_drift = float(np.mean(np.abs(pred_ages_arr - clean_ref_preds["pred_ages"])))\n        g_consistency = float(np.mean(((pred_g_arr >= 0.5) == (clean_ref_preds["pred_g"] >= 0.5))))\n        p1 = np.clip(clean_ref_preds["pred_ldl"], 1e-12, 1.0)\n        p2 = np.clip(pred_ldl_arr, 1e-12, 1.0)\n        m = 0.5 * (p1 + p2)\n        js = 0.5 * np.sum(p1 * (np.log(p1) - np.log(m)), axis=1) + 0.5 * np.sum(p2 * (np.log(p2) - np.log(m)), axis=1)\n        metrics["age_drift_vs_clean_years"] = round(age_drift, 4)\n        metrics["gender_agreement_vs_clean"] = round(g_consistency, 4)\n        metrics["mean_js_divergence_vs_clean"] = round(float(np.mean(js)), 5)\n\n    raw_arrays = {\n        "pred_ages": pred_ages_arr,\n        "true_ages": true_ages_arr,\n        "valid_ages": valid_ages_arr,\n        "pred_g": pred_g_arr,\n        "true_g": true_g_arr,\n        "valid_g": valid_g_arr,\n        "pred_ldl": pred_ldl_arr,\n        "pred_iv": pred_iv_arr,\n        "true_iv": true_iv_arr,\n        "total_stds": total_std_arr,\n        "ita_groups": np.array(ita_groups),\n        "sources": np.array(sources),\n    }\n    return metrics, raw_arrays\n\ndef evaluate_demographic_fairness(raw_arrays: Dict[str, np.ndarray]) -> Dict[str, Any]:\n    pred_ages = raw_arrays["pred_ages"]\n    true_ages = raw_arrays["true_ages"]\n    valid_ages = raw_arrays["valid_ages"]\n    pred_g = raw_arrays["pred_g"]\n    true_g = raw_arrays["true_g"]\n    valid_g = raw_arrays["valid_g"]\n    ita_groups = raw_arrays["ita_groups"]\n\n    ita_breakdown = {}\n    ita_maes, ita_g_accs = [], []\n    for grp in sorted(set(ita_groups.tolist())):\n        mask = (ita_groups == grp)\n        cnt = int(np.sum(mask))\n        if cnt == 0: continue\n        m_age = mask & valid_ages\n        m_g = mask & valid_g\n        mae_g = float(np.mean(np.abs(pred_ages[m_age] - true_ages[m_age]))) if np.any(m_age) else 0.0\n        g_acc = float(np.mean((pred_g[m_g] >= 0.5).astype(np.int64) == true_g[m_g])) if np.any(m_g) else 0.0\n        ita_breakdown[grp] = {\n            "count": cnt,\n            "age_mae": round(mae_g, 4),\n            "gender_accuracy": round(g_acc, 4),\n        }\n        if cnt >= 5:\n            ita_maes.append(mae_g)\n            ita_g_accs.append(g_acc)\n\n    return {\n        "ita_skin_tone_groups": ita_breakdown,\n        "ita_age_mae_std_across_groups": round(float(np.std(ita_maes)), 4) if ita_maes else 0.0,\n        "ita_gender_acc_std_across_groups": round(float(np.std(ita_g_accs)), 4) if ita_g_accs else 0.0,\n    }\n\ndef evaluate_calibration_and_cohorts(raw_arrays: Dict[str, np.ndarray]) -> Dict[str, Any]:\n    pred_ages = raw_arrays["pred_ages"]\n    true_ages = raw_arrays["true_ages"]\n    valid_ages = raw_arrays["valid_ages"]\n    pred_g = raw_arrays["pred_g"]\n    true_g = raw_arrays["true_g"]\n    valid_g = raw_arrays["valid_g"]\n    pred_ldl = raw_arrays["pred_ldl"]\n    sources = raw_arrays["sources"]\n\n    vg_p = pred_g[valid_g]\n    vg_y = true_g[valid_g]\n    brier = float(np.mean((vg_p - vg_y) ** 2)) if len(vg_y) > 0 else 0.0\n    ece = 0.0\n    if len(vg_y) > 0:\n        conf = np.maximum(vg_p, 1.0 - vg_p)\n        correct = ((vg_p >= 0.5).astype(np.int64) == vg_y).astype(np.float64)\n        bins = np.linspace(0.5, 1.0, 11)\n        for bi in range(10):\n            bm = (conf >= bins[bi]) & (conf < bins[bi + 1] if bi < 9 else conf <= bins[bi + 1])\n            if np.any(bm):\n                ece += (np.sum(bm) / len(vg_y)) * abs(float(np.mean(correct[bm])) - float(np.mean(conf[bm])))\n\n    # 90% credible interval coverage from 101-bin LDL CDF (5th to 95th percentile)\n    va_ldl = pred_ldl[valid_ages]\n    va_true = true_ages[valid_ages]\n    cdfs = np.cumsum(va_ldl, axis=1)\n    q05 = np.argmax(cdfs >= 0.05, axis=1).astype(np.float64)\n    q95 = np.argmax(cdfs >= 0.95, axis=1).astype(np.float64)\n    cov_90 = float(np.mean((va_true >= (q05 - 1.5)) & (va_true <= (q95 + 1.5)))) if len(va_true) > 0 else 0.0\n\n    source_breakdown = {}\n    for src in sorted(set(sources.tolist())):\n        sm = (sources == src)\n        cnt = int(np.sum(sm))\n        if cnt == 0: continue\n        sm_a = sm & valid_ages\n        sm_g = sm & valid_g\n        source_breakdown[src] = {\n            "count": cnt,\n            "age_mae": round(float(np.mean(np.abs(pred_ages[sm_a] - true_ages[sm_a]))), 4) if np.any(sm_a) else 0.0,\n            "gender_accuracy": round(float(np.mean((pred_g[sm_g] >= 0.5).astype(np.int64) == true_g[sm_g])), 4) if np.any(sm_g) else 0.0,\n        }\n\n    return {\n        "gender_expected_calibration_error": round(float(ece), 4),\n        "gender_brier_score": round(brier, 4),\n        "age_ldl_90pct_credible_interval_coverage": round(cov_90, 4),\n        "per_dataset_source_breakdown": source_breakdown,\n    }\n\n@torch.no_grad()\ndef benchmark_smoothness_and_latency(model: BiometricMultiTaskModel, test_dataset: FaceGoldenDataset) -> Dict[str, Any]:\n    model.eval()\n    # 1. Latency benchmark (Warmup + 50 iterations for B=1 single-pass, B=1 Flip-TTA, and B=32 batched at canonical 224x224)\n    x1 = torch.randn(1, 3, 224, 224, device=device)\n    x32 = torch.randn(32, 3, 224, 224, device=device)\n    for _ in range(10):\n        _ = model(x1, return_dict=True)\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n    t0 = time.perf_counter()\n    for _ in range(50):\n        _ = model(x1, return_dict=True)\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n    lat_b1_ms = ((time.perf_counter() - t0) / 50.0) * 1000.0\n\n    t0 = time.perf_counter()\n    for _ in range(50):\n        _ = model.predict_smooth(x1, enable_flip_tta=True)\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n    lat_tta_ms = ((time.perf_counter() - t0) / 50.0) * 1000.0\n\n    t0 = time.perf_counter()\n    for _ in range(30):\n        _ = model(x32, return_dict=True)\n    if torch.cuda.is_available(): torch.cuda.synchronize()\n    lat_b32_per_face_ms = (((time.perf_counter() - t0) / 30.0) * 1000.0) / 32.0\n\n    # 2. Temporal Stream Smoothness Benchmark across 24 test identities x 20 frames with environmental corruptions\n    num_tracks = min(24, len(test_dataset))\n    num_frames = 20\n    regimes_cycle = ExtremeNoiseStressBenchmark.REGIMES\n    raw_age_jitters, locked_age_jitters = [], []\n    raw_iv_flickers, locked_iv_flickers = [], []\n    raw_gender_flips, locked_gender_flips = [], []\n\n    for tid in range(num_tracks):\n        base_rgb = test_dataset.images_rgb[tid]\n        frames_t = []\n        for f in range(num_frames):\n            reg = regimes_cycle[f % len(regimes_cycle)]\n            c_rgb = ExtremeNoiseStressBenchmark.apply_regime_uint8(base_rgb, regime=reg, seed=5000 + tid * 100 + f)\n            frames_t.append(torch.from_numpy(c_rgb.transpose(2, 0, 1)).float() / 127.5 - 1.0)\n        batch_t = torch.stack(frames_t, dim=0).to(device)\n        out = model.predict_smooth(batch_t, enable_flip_tta=True)\n        r_ages = out["expected_age"].cpu().numpy()\n        r_stds = out["total_std"].cpu().numpy()\n        r_g_logits = out["gender_logit"].cpu().numpy()\n        r_ivs = out["age_interval_idx"].cpu().numpy()\n        r_g_labels = ["Female" if g >= 0.0 else "Male" for g in r_g_logits]\n\n        locker = TemporalTargetLocker(initial_age=float(r_ages[0]), initial_std=float(r_stds[0]), initial_logit=float(r_g_logits[0]))\n        l_ages, l_ivs, l_g_labels = [float(r_ages[0])], [int(locker.interval_idx)], [r_g_labels[0]]\n        for f in range(1, num_frames):\n            la, liv, lg = locker.update(float(r_ages[f]), float(r_stds[f]), float(r_g_logits[f]))\n            l_ages.append(la)\n            l_ivs.append(liv)\n            l_g_labels.append(lg)\n\n        raw_age_jitters.append(float(np.std(np.diff(r_ages))))\n        locked_age_jitters.append(float(np.std(np.diff(np.array(l_ages)))))\n        raw_iv_flickers.append(float(np.mean(np.diff(r_ivs) != 0)))\n        locked_iv_flickers.append(float(np.mean(np.diff(np.array(l_ivs)) != 0)))\n        raw_gender_flips.append(float(np.mean([r_g_labels[i] != r_g_labels[i - 1] for i in range(1, num_frames)])))\n        locked_gender_flips.append(float(np.mean([l_g_labels[i] != l_g_labels[i - 1] for i in range(1, num_frames)])))\n\n    raw_jitter_mean = float(np.mean(raw_age_jitters))\n    locked_jitter_mean = float(np.mean(locked_age_jitters))\n    raw_flicker_mean = float(np.mean(raw_iv_flickers))\n    locked_flicker_mean = float(np.mean(locked_iv_flickers))\n\n    return {\n        "canonical_resolution": "224x224",\n        "single_face_latency_ms": round(lat_b1_ms, 3),\n        "single_face_flip_tta_latency_ms": round(lat_tta_ms, 3),\n        "batched_b32_per_face_latency_ms": round(lat_b32_per_face_ms, 3),\n        "throughput_faces_per_sec_b32": round(1000.0 / max(1e-4, lat_b32_per_face_ms), 1),\n        "temporal_stream_raw_age_jitter_years": round(raw_jitter_mean, 4),\n        "temporal_stream_locked_age_jitter_years": round(locked_jitter_mean, 4),\n        "temporal_age_jitter_reduction_pct": round(100.0 * (1.0 - locked_jitter_mean / max(1e-6, raw_jitter_mean)), 2),\n        "temporal_stream_raw_23_interval_flicker_rate": round(raw_flicker_mean, 4),\n        "temporal_stream_locked_23_interval_flicker_rate": round(locked_flicker_mean, 4),\n        "temporal_interval_flicker_reduction_pct": round(100.0 * (1.0 - locked_flicker_mean / max(1e-6, raw_flicker_mean)), 2),\n        "temporal_stream_raw_gender_flip_rate": round(float(np.mean(raw_gender_flips)), 4),\n        "temporal_stream_locked_gender_flip_rate": round(float(np.mean(locked_gender_flips)), 4),\n    }\n\n@torch.no_grad()\ndef precompute_teacher_targets_and_refine_dataset(\n    teacher_model: BiometricMultiTaskModel,\n    train_ds: FaceGoldenDataset,\n    batch_size: int = 64,\n    gender_conf_thresh: float = 0.80,\n) -> Dict[str, Any]:\n    teacher_model.eval()\n    N = len(train_ds)\n    refined_intervals_cnt = 0\n    imputed_gender_cnt = 0\n\n    for start in range(0, N, batch_size):\n        end = min(N, start + batch_size)\n        batch_imgs = []\n        for idx in range(start, end):\n            raw_rgb = train_ds.images_rgb[idx]\n            t_img = torch.from_numpy(raw_rgb.transpose(2, 0, 1)).float() / 127.5 - 1.0\n            batch_imgs.append(t_img)\n        x_t = torch.stack(batch_imgs, dim=0).to(device)\n        out = teacher_model.predict_smooth(x_t, enable_flip_tta=True)\n\n        t_age_logits = out["age_logits"].detach().cpu()\n        t_age_probs = out["age_probs"].detach().cpu()\n        t_iv_logits = out["fine_interval_logits"].detach().cpu()\n        t_g_logits = out["gender_logit"].detach().cpu()\n        t_reid = out["reid_embedding"].detach().cpu()\n\n        b_len = end - start\n        box_101 = torch.stack([train_ds.cached_targets[start + j]["ldl_101"] for j in range(b_len)], dim=0)\n        a_min = torch.stack([train_ds.cached_targets[start + j]["age_min"] for j in range(b_len)], dim=0)\n        a_max = torch.stack([train_ds.cached_targets[start + j]["age_max"] for j in range(b_len)], dim=0)\n        ref_101, ref_23 = refine_interval_ldl_with_teacher(box_101, t_age_probs, a_min, a_max, blend_alpha=0.65)\n\n        for j in range(b_len):\n            ct = train_ds.cached_targets[start + j]\n            span = float((ct["age_max"] - ct["age_min"]).item())\n            if span >= 2.0 and float(ct["has_valid_age"].item()) > 0.5:\n                ct["ldl_101"] = ref_101[j]\n                ct["ldl_fine_23"] = ref_23[j]\n                bins_101 = torch.arange(101, dtype=torch.float32)\n                ref_exp = float(torch.sum(ref_101[j] * bins_101).item())\n                ct["age_continuous"] = torch.tensor(float(np.clip(ref_exp, float(ct["age_min"].item()), float(ct["age_max"].item()))), dtype=torch.float32)\n                refined_intervals_cnt += 1\n\n            g_prob_j = float(torch.sigmoid(t_g_logits[j]).item())\n            g_conf_j = max(g_prob_j, 1.0 - g_prob_j)\n            if float(ct["gender_mask"].item()) < 0.5 and g_conf_j >= gender_conf_thresh:\n                imputed_gender_cnt += 1\n\n            ct["teacher_age_logits"] = t_age_logits[j]\n            ct["teacher_age_probs"] = t_age_probs[j]\n            ct["teacher_fine_interval_logits"] = t_iv_logits[j]\n            ct["teacher_gender_logit"] = t_g_logits[j]\n            ct["teacher_reid_embedding"] = t_reid[j]\n\n    return {\n        "total_train_samples": N,\n        "refined_coarse_interval_samples": refined_intervals_cnt,\n        "teacher_imputed_missing_gender_samples": imputed_gender_cnt,\n    }\n\n# ---------------------------------------------------------------------------\n# 9. Main Training & Evaluation Loop (Two-Stage Teacher-Student Distillation + Model Soup)\n# ---------------------------------------------------------------------------\ndef run_training_and_evaluation():\n    t_start = time.time()\n    manifest_p = discover_golden_manifest()\n    if is_main_process:\n        print(f"📂 Discovered Curated Golden Dataset Manifest: {manifest_p}")\n\n    train_ds = FaceGoldenDataset(manifest_p, split="train", is_train=True)\n    val_ds = FaceGoldenDataset(manifest_p, split="val", is_train=False)\n    test_ds = FaceGoldenDataset(manifest_p, split="test", is_train=False)\n\n    # Zero-Leakage & Provenance Verification\n    train_subj = set(train_ds.df["subject_id"].dropna().astype(str))\n    val_subj = set(val_ds.df["subject_id"].dropna().astype(str))\n    test_subj = set(test_ds.df["subject_id"].dropna().astype(str))\n    leak_tv = len(train_subj & val_subj)\n    leak_tt = len(train_subj & test_subj)\n    leak_vt = len(val_subj & test_subj)\n    assert leak_tv == 0 and leak_tt == 0 and leak_vt == 0, f"Cross-split identity leakage detected: {leak_tv}, {leak_tt}, {leak_vt}"\n    if is_main_process:\n        print(f"🔒 Zero-Leakage Verified: Train={len(train_ds)} ({len(train_subj)} subj) | Val={len(val_ds)} ({len(val_subj)} subj) | Test={len(test_ds)} ({len(test_subj)} subj)")\n\n    criterion = UnifiedBiometricLoss(\n        w_cramer=1.0, w_kl=0.5, w_mae=0.25, w_interval=0.50,\n        w_fine_23=0.80, w_smoothness=0.15, w_gender=1.2, w_unc=0.1, w_reid=0.15,\n    )\n\n    # =========================================================================\n    # STAGE A: High-Capacity Teacher Training (convnextv2_base, 88.5M params)\n    # =========================================================================\n    teacher_batch_size = 16\n    if use_ddp:\n        t_sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=local_rank, shuffle=True)\n        t_loader = DataLoader(train_ds, batch_size=teacher_batch_size, sampler=t_sampler, num_workers=0, pin_memory=True, drop_last=False)\n    else:\n        t_sampler = None\n        t_loader = DataLoader(train_ds, batch_size=teacher_batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=False)\n\n    teacher_base = BiometricMultiTaskModel(backbone_name="convnextv2_base", pretrained=True, drop_path_rate=0.20).to(device)\n    teacher_ema = ModelEMA(teacher_base, decay=0.990)\n    teacher_params_m = sum(p.numel() for p in teacher_base.parameters()) / 1e6\n\n    if use_ddp:\n        teacher_model = DDP(teacher_base, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False, gradient_as_bucket_view=False)\n    else:\n        teacher_model = teacher_base\n\n    t_bb_params, t_head_params = [], []\n    for name, p in teacher_base.named_parameters():\n        if not p.requires_grad: continue\n        if name.startswith("backbone."):\n            t_bb_params.append(p)\n        else:\n            t_head_params.append(p)\n\n    t_opt = torch.optim.AdamW(\n        [\n            {"params": t_bb_params, "lr": 1.2e-4, "base_lr": 1.2e-4},\n            {"params": t_head_params, "lr": 5.0e-4, "base_lr": 5.0e-4},\n        ],\n        weight_decay=1e-2,\n    )\n    teacher_epochs = 16\n    t_steps_per_ep = max(1, len(t_loader))\n    t_warmup = 2 * t_steps_per_ep\n    t_total_steps = teacher_epochs * t_steps_per_ep\n    t_sched = torch.optim.lr_scheduler.CosineAnnealingLR(t_opt, T_max=max(1, t_total_steps - t_warmup), eta_min=2e-6)\n    teacher_ckpt_path = Path("convnextv2_base_teacher.pt")\n    teacher_resume_path = Path("teacher_resume_state.pt")\n    student_resume_path = Path("student_resume_state.pt")\n    max_runtime_sec = float(os.environ.get("BIOMETRIC_MAX_RUNTIME_SEC", "28800"))\n    stage_a_budget_sec = float(os.environ.get("BIOMETRIC_STAGE_A_BUDGET_SEC", "10800"))\n    use_amp = torch.cuda.is_available()\n    t_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)\n    best_t_score = float("inf")\n    t_start_ep = 0\n    t_step = 0\n\n    for cand_t_res in [teacher_resume_path, Path("/kaggle/input/face-convnext-v2-biometric-training-pipeline/teacher_resume_state.pt")]:\n        if cand_t_res.exists():\n            t_st = torch.load(cand_t_res, map_location=device)\n            teacher_base.load_state_dict(t_st["model"])\n            teacher_ema.ema.load_state_dict(t_st["ema"])\n            t_opt.load_state_dict(t_st["opt"])\n            t_sched.load_state_dict(t_st["sched"])\n            best_t_score = float(t_st.get("best_t_score", float("inf")))\n            t_start_ep = int(t_st.get("epoch", 0))\n            t_step = int(t_st.get("step", 0))\n            if is_main_process:\n                print(f"🔄 [Stage A Resume] Resumed Teacher from {cand_t_res} at epoch {t_start_ep}/{teacher_epochs}", flush=True)\n            break\n\n    if is_main_process:\n        print(f"\\n🎓 [Stage A] Training High-Capacity Teacher (convnextv2_base | {teacher_params_m:.2f}M params | 224x224 | AMP={use_amp}) for {teacher_epochs} epochs...", flush=True)\n\n    for ep in range(t_start_ep, teacher_epochs):\n        t_ep0 = time.time()\n        if t_sampler is not None:\n            t_sampler.set_epoch(ep)\n        teacher_model.train()\n        ep_loss = 0.0\n        ep_steps = 0\n        for batch in t_loader:\n            t_step += 1\n            ep_steps += 1\n            imgs_clean = batch["image"].to(device, non_blocking=True)\n            imgs_corrupt = batch["image_corrupted"].to(device, non_blocking=True)\n            targets = {\n                k: v.to(device, non_blocking=True)\n                for k, v in batch.items()\n                if isinstance(v, torch.Tensor) and k not in {"image", "image_corrupted", "sample_idx"}\n            }\n            if t_step <= t_warmup:\n                wf = float(t_step) / float(max(1, t_warmup))\n                for pg in t_opt.param_groups:\n                    pg["lr"] = pg["base_lr"] * wf\n            t_opt.zero_grad(set_to_none=True)\n            bsz = imgs_clean.shape[0]\n            with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16):\n                preds_both = teacher_model(torch.cat([imgs_clean, imgs_corrupt], dim=0))\n            preds_c = {k: v[:bsz].float() if isinstance(v, torch.Tensor) else v for k, v in preds_both.items() if isinstance(v, torch.Tensor)}\n            preds_n = {k: v[bsz:].float() if isinstance(v, torch.Tensor) else v for k, v in preds_both.items() if isinstance(v, torch.Tensor)}\n            l_c, _ = criterion(preds_c, targets)\n            l_n, _ = criterion(preds_n, targets)\n            l_cons = photometric_consistency_loss(preds_c, preds_n, age_mask=targets["has_valid_age"], gender_mask=targets["gender_mask"])\n            loss_t = 0.55 * l_c + 0.45 * l_n + 0.30 * l_cons\n            t_scaler.scale(loss_t).backward()\n            t_scaler.unscale_(t_opt)\n            torch.nn.utils.clip_grad_norm_(teacher_model.parameters(), max_norm=5.0)\n            t_scaler.step(t_opt)\n            t_scaler.update()\n            teacher_ema.update(teacher_base)\n            if t_step > t_warmup:\n                t_sched.step()\n            ep_loss += float(loss_t.item())\n\n        if is_main_process and ((ep + 1) % 2 == 0 or ep == 0 or (ep + 1) == teacher_epochs):\n            vm_raw, _ = evaluate_dataset_regime(teacher_base, val_ds, regime="clean", enable_flip_tta=True, batch_size=32)\n            vm_ema, _ = evaluate_dataset_regime(teacher_ema.ema, val_ds, regime="clean", enable_flip_tta=True, batch_size=32)\n            sr = vm_raw["age_mae"] - 8.0 * vm_raw["gender_accuracy"] - 5.0 * vm_raw["fine_23_interval_within_1_bracket_acc"] - 3.0 * vm_raw["fine_23_interval_exact_acc"]\n            se = vm_ema["age_mae"] - 8.0 * vm_ema["gender_accuracy"] - 5.0 * vm_ema["fine_23_interval_within_1_bracket_acc"] - 3.0 * vm_ema["fine_23_interval_exact_acc"]\n            use_te = se <= sr\n            vm = vm_ema if use_te else vm_raw\n            sc = min(sr, se)\n            mk = ""\n            if sc < best_t_score:\n                best_t_score = sc\n                torch.save(teacher_ema.ema.state_dict() if use_te else teacher_base.state_dict(), teacher_ckpt_path)\n                mk = " ⭐ [BEST TEACHER]"\n            torch.save({\n                "epoch": ep + 1,\n                "step": t_step,\n                "best_t_score": best_t_score,\n                "model": teacher_base.state_dict(),\n                "ema": teacher_ema.ema.state_dict(),\n                "opt": t_opt.state_dict(),\n                "sched": t_sched.state_dict(),\n            }, teacher_resume_path)\n            print(\n                f"  [Teacher Ep {ep+1:02d}/{teacher_epochs} ({time.time()-t_ep0:.1f}s)] Loss: {ep_loss/max(1,ep_steps):.4f} | "\n                f"Val MAE: {vm[\'age_mae\']:.2f}y | 23-Int(Exact/±1): {vm[\'fine_23_interval_exact_acc\']*100:.1f}%/{vm[\'fine_23_interval_within_1_bracket_acc\']*100:.1f}% | "\n                f"Gender Acc: {vm[\'gender_accuracy\']*100:.1f}%{mk}",\n                flush=True,\n            )\n\n        # Synchronized Stage-A wall-clock budget check across DDP ranks\n        stop_stage_a = torch.tensor([1 if (time.time() - t_start) >= stage_a_budget_sec else 0], dtype=torch.int32, device=device)\n        if use_ddp and dist.is_initialized():\n            dist.broadcast(stop_stage_a, src=0)\n        if int(stop_stage_a.item()) == 1:\n            if is_main_process:\n                print(f"⏱️ [Stage A Time Budget Guard] Elapsed {time.time() - t_start:.1f}s >= {stage_a_budget_sec:.1f}s; advancing to Stage B distillation with best Teacher checkpoint.", flush=True)\n            break\n\n    if use_ddp and dist.is_initialized():\n        dist.barrier()\n\n    if teacher_ckpt_path.exists():\n        teacher_ema.ema.load_state_dict(torch.load(teacher_ckpt_path, map_location=device))\n    teacher_ema.ema.eval()\n\n    teacher_val_metrics, _ = evaluate_dataset_regime(teacher_ema.ema, val_ds, regime="clean", enable_flip_tta=True, batch_size=32) if is_main_process else ({}, {})\n    teacher_test_metrics, _ = evaluate_dataset_regime(teacher_ema.ema, test_ds, regime="clean", enable_flip_tta=True, batch_size=32) if is_main_process else ({}, {})\n\n    distill_prep_stats = precompute_teacher_targets_and_refine_dataset(teacher_ema.ema, train_ds, batch_size=32, gender_conf_thresh=0.80)\n    if is_main_process:\n        print(\n            f"✨ [Distillation Target Refinement] Teacher refined {distill_prep_stats[\'refined_coarse_interval_samples\']} coarse-interval LDL targets "\n            f"and imputed {distill_prep_stats[\'teacher_imputed_missing_gender_samples\']} high-confidence missing-gender pseudo-labels.",\n            flush=True,\n        )\n        for p_clean in [teacher_ckpt_path, teacher_resume_path]:\n            if p_clean.exists():\n                try:\n                    p_clean.unlink()\n                except Exception:\n                    pass\n\n    del teacher_model, teacher_base, teacher_ema, t_opt, t_sched\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n    # =========================================================================\n    # STAGE B: Student Distillation Training (convnextv2_tiny, 28.6M params)\n    # =========================================================================\n    batch_size = 32\n    if use_ddp:\n        sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=local_rank, shuffle=True)\n        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0, pin_memory=True, drop_last=False)\n    else:\n        sampler = None\n        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=False)\n\n    base_model = BiometricMultiTaskModel(backbone_name="convnextv2_tiny", pretrained=True, drop_path_rate=0.15).to(device)\n    ema = ModelEMA(base_model, decay=0.992)\n\n    if use_ddp:\n        model = DDP(base_model, device_ids=[local_rank], output_device=local_rank, find_unused_parameters=False, gradient_as_bucket_view=False)\n    else:\n        model = base_model\n\n    distill_criterion = BiometricDistillationLoss(\n        temperature=2.0,\n        w_age_kl=0.85,\n        w_age_cramer=0.65,\n        w_interval_kl=0.65,\n        w_gender=0.90,\n        w_reid=0.45,\n        gender_conf_thresh=0.80,\n    )\n\n    # Layer-wise differential learning rates\n    backbone_params, head_params = [], []\n    for name, p in base_model.named_parameters():\n        if not p.requires_grad: continue\n        if name.startswith("backbone."):\n            backbone_params.append(p)\n        else:\n            head_params.append(p)\n\n    base_lr_backbone = 1.8e-4\n    base_lr_heads = 6.5e-4\n    optimizer = torch.optim.AdamW(\n        [\n            {"params": backbone_params, "lr": base_lr_backbone, "base_lr": base_lr_backbone},\n            {"params": head_params, "lr": base_lr_heads, "base_lr": base_lr_heads},\n        ],\n        weight_decay=1e-2,\n    )\n\n    num_epochs = 36\n    steps_per_epoch = max(1, len(train_loader))\n    warmup_steps = 3 * steps_per_epoch\n    total_steps = num_epochs * steps_per_epoch\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps), eta_min=2e-6)\n    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)\n\n    global_step = 0\n    start_epoch = 0\n    initial_loss = None\n    best_val_score = float("inf")\n    best_epoch = 0\n    history: List[Dict[str, Any]] = []\n    top_checkpoints: List[Tuple[float, Dict[str, torch.Tensor]]] = []\n    ckpt_path = Path("convnextv2_tiny_biometrics.pt")\n    report_path = Path("training_evaluation_report.json")\n\n    for cand_s_res in [student_resume_path, Path("/kaggle/input/face-convnext-v2-biometric-training-pipeline/student_resume_state.pt")]:\n        if cand_s_res.exists():\n            s_st = torch.load(cand_s_res, map_location=device)\n            base_model.load_state_dict(s_st["model"])\n            ema.ema.load_state_dict(s_st["ema"])\n            optimizer.load_state_dict(s_st["opt"])\n            scheduler.load_state_dict(s_st["sched"])\n            start_epoch = int(s_st.get("epoch", 0))\n            global_step = int(s_st.get("global_step", 0))\n            best_val_score = float(s_st.get("best_val_score", float("inf")))\n            best_epoch = int(s_st.get("best_epoch", 0))\n            history = list(s_st.get("history", []))\n            if is_main_process:\n                print(f"🔄 [Stage B Resume] Resumed Student from {cand_s_res} at epoch {start_epoch}/{num_epochs}", flush=True)\n            break\n\n    if is_main_process:\n        total_params = sum(p.numel() for p in base_model.parameters())\n        print(f"\\n🚀 [Stage B] Starting Knowledge-Distilled ConvNeXt-V2-Tiny Training | Params: {total_params / 1e6:.2f}M | Resolution: 224x224 | AMP={use_amp} | Epochs: {num_epochs} | Steps/Epoch: {steps_per_epoch}", flush=True)\n\n    teacher_keys = {\n        "teacher_age_logits",\n        "teacher_age_probs",\n        "teacher_fine_interval_logits",\n        "teacher_gender_logit",\n        "teacher_reid_embedding",\n    }\n\n    for epoch in range(start_epoch, num_epochs):\n        t_ep = time.time()\n        if sampler is not None:\n            sampler.set_epoch(epoch)\n        model.train()\n        epoch_loss_sum = 0.0\n        epoch_steps = 0\n\n        for batch in train_loader:\n            global_step += 1\n            epoch_steps += 1\n            imgs_clean = batch["image"].to(device, non_blocking=True)\n            imgs_corrupt = batch["image_corrupted"].to(device, non_blocking=True)\n            targets = {\n                k: v.to(device, non_blocking=True)\n                for k, v in batch.items()\n                if isinstance(v, torch.Tensor) and k not in ({"image", "image_corrupted", "sample_idx"} | teacher_keys)\n            }\n            teacher_batch = {\n                k: batch[k].to(device, non_blocking=True)\n                for k in teacher_keys\n                if k in batch and isinstance(batch[k], torch.Tensor)\n            }\n\n            if global_step <= warmup_steps:\n                w_frac = float(global_step) / float(max(1, warmup_steps))\n                for pg in optimizer.param_groups:\n                    pg["lr"] = pg["base_lr"] * w_frac\n\n            optimizer.zero_grad(set_to_none=True)\n\n            # Dual-view forward pass with FP16 Tensor Core AMP on backbone + FP32 stem & loss\n            bsz = imgs_clean.shape[0]\n            imgs_both = torch.cat([imgs_clean, imgs_corrupt], dim=0)\n            with torch.cuda.amp.autocast(enabled=use_amp, dtype=torch.float16):\n                preds_both = model(imgs_both)\n            preds_clean = {k: v[:bsz].float() if isinstance(v, torch.Tensor) else v for k, v in preds_both.items() if isinstance(v, torch.Tensor)}\n            preds_corrupt = {k: v[bsz:].float() if isinstance(v, torch.Tensor) else v for k, v in preds_both.items() if isinstance(v, torch.Tensor)}\n\n            loss_clean, metrics = criterion(preds_clean, targets)\n            loss_corrupt, _ = criterion(preds_corrupt, targets)\n            loss_cons = photometric_consistency_loss(\n                preds_clean, preds_corrupt,\n                age_mask=targets["has_valid_age"],\n                gender_mask=targets["gender_mask"],\n            )\n            loss = 0.55 * loss_clean + 0.45 * loss_corrupt + 0.35 * loss_cons\n\n            if len(teacher_batch) == len(teacher_keys):\n                kd_clean, _ = distill_criterion(preds_clean, teacher_batch, targets=targets)\n                kd_corrupt, _ = distill_criterion(preds_corrupt, teacher_batch, targets=targets)\n                loss = loss + 0.45 * (0.60 * kd_clean + 0.40 * kd_corrupt)\n\n            # FAIL-FAST TRIPWIRE 1: NaN/Inf loss check\n            if torch.isnan(loss) or torch.isinf(loss):\n                raise RuntimeError(f"🚨 [FAIL-FAST TRIPWIRE 1] Loss is NaN/Inf at step {global_step}!")\n\n            # FAIL-FAST TRIPWIRE 2: Divergence check at step 100\n            if global_step == 1:\n                initial_loss = float(loss.item())\n            elif global_step == 100 and initial_loss is not None:\n                if float(loss.item()) > 2.0 * initial_loss:\n                    raise RuntimeError(f"🚨 [FAIL-FAST TRIPWIRE 2] Loss diverged at step 100: {loss.item():.4f} > 2.0x {initial_loss:.4f}!")\n\n            # FAIL-FAST TRIPWIRE 3: Variance explosion/collapse check\n            mean_log_var = float(preds_clean["log_variance"].mean().item())\n            if mean_log_var > 6.0 or mean_log_var < -9.0:\n                raise RuntimeError(f"🚨 [FAIL-FAST TRIPWIRE 3] Variance anomaly (mean log_var={mean_log_var:.2f}) at step {global_step}!")\n\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)\n            if torch.isnan(grad_norm) or torch.isinf(grad_norm):\n                raise RuntimeError(f"🚨 [FAIL-FAST TRIPWIRE 1B] NaN/Inf gradient norm at step {global_step}!")\n\n            scaler.step(optimizer)\n            scaler.update()\n            ema.update(base_model)\n\n            if global_step > warmup_steps:\n                scheduler.step()\n\n            epoch_loss_sum += float(loss.item())\n\n        # Per-epoch Validation & Incremental Checkpointing on rank 0\n        if is_main_process and ((epoch + 1) % 2 == 0 or epoch == 0 or (epoch + 1) == num_epochs or (epoch + 1) >= 20):\n            # Evaluate both raw base_model and EMA model; select whichever has better composite validation score\n            val_metrics_raw, _ = evaluate_dataset_regime(base_model, val_ds, regime="clean", enable_flip_tta=True, batch_size=32)\n            val_metrics_ema, _ = evaluate_dataset_regime(ema.ema, val_ds, regime="clean", enable_flip_tta=True, batch_size=32)\n\n            score_raw = val_metrics_raw["age_mae"] - 8.0 * val_metrics_raw["gender_accuracy"] - 5.0 * val_metrics_raw["fine_23_interval_within_1_bracket_acc"] - 3.0 * val_metrics_raw["fine_23_interval_exact_acc"]\n            score_ema = val_metrics_ema["age_mae"] - 8.0 * val_metrics_ema["gender_accuracy"] - 5.0 * val_metrics_ema["fine_23_interval_within_1_bracket_acc"] - 3.0 * val_metrics_ema["fine_23_interval_exact_acc"]\n            use_ema_best = score_ema <= score_raw\n            val_m = val_metrics_ema if use_ema_best else val_metrics_raw\n            val_score = min(score_raw, score_ema)\n            cur_lr = optimizer.param_groups[0]["lr"]\n\n            epoch_rec = {\n                "epoch": epoch + 1,\n                "train_loss": round(epoch_loss_sum / max(1, epoch_steps), 4),\n                "val_source": "ema" if use_ema_best else "model",\n                "val_age_mae": val_m["age_mae"],\n                "val_cs_at_5": val_m["cs_at_5"],\n                "val_fine_23_exact_acc": val_m["fine_23_interval_exact_acc"],\n                "val_fine_23_within_1_acc": val_m["fine_23_interval_within_1_bracket_acc"],\n                "val_gender_acc": val_m["gender_accuracy"],\n                "val_gender_auc": val_m["gender_roc_auc"],\n            }\n            history.append(epoch_rec)\n\n            cand_state = {k: v.detach().cpu().clone() for k, v in (ema.ema.state_dict() if use_ema_best else base_model.state_dict()).items()}\n            top_checkpoints.append((val_score, cand_state))\n            top_checkpoints.sort(key=lambda x: x[0])\n            if len(top_checkpoints) > 3:\n                top_checkpoints.pop()\n\n            marker = ""\n            if val_score < best_val_score:\n                best_val_score = val_score\n                best_epoch = epoch + 1\n                torch.save(cand_state, ckpt_path)\n                marker = " ⭐ [BEST]"\n\n            # Save resumable student state + live evaluation snapshot after every validation pass\n            torch.save({\n                "epoch": epoch + 1,\n                "global_step": global_step,\n                "best_val_score": best_val_score,\n                "best_epoch": best_epoch,\n                "history": history,\n                "model": base_model.state_dict(),\n                "ema": ema.ema.state_dict(),\n                "opt": optimizer.state_dict(),\n                "sched": scheduler.state_dict(),\n            }, student_resume_path)\n\n            live_snapshot = {\n                "status": "in_progress_checkpoint",\n                "last_completed_epoch": epoch + 1,\n                "best_epoch": best_epoch,\n                "validation_clean_metrics": val_m,\n                "epoch_history": history,\n            }\n            report_path.write_text(json.dumps(live_snapshot, indent=2), encoding="utf-8")\n\n            print(\n                f"[Epoch {epoch+1:02d}/{num_epochs} ({time.time()-t_ep:.1f}s)] TrainLoss: {epoch_rec[\'train_loss\']:.4f} | "\n                f"Val MAE: {val_m[\'age_mae\']:.2f}y | CS@5: {val_m[\'cs_at_5\']*100:.1f}% | "\n                f"23-Int(Exact/±1): {val_m[\'fine_23_interval_exact_acc\']*100:.1f}%/{val_m[\'fine_23_interval_within_1_bracket_acc\']*100:.1f}% | "\n                f"Gender Acc: {val_m[\'gender_accuracy\']*100:.1f}% (AUC {val_m[\'gender_roc_auc\']:.3f}) | "\n                f"LR: {cur_lr:.2e}{marker}",\n                flush=True,\n            )\n\n        # Synchronized Stage-B wall-clock budget check across DDP ranks\n        stop_stage_b = torch.tensor([1 if (time.time() - t_start) >= max_runtime_sec else 0], dtype=torch.int32, device=device)\n        if use_ddp and dist.is_initialized():\n            dist.broadcast(stop_stage_b, src=0)\n        if int(stop_stage_b.item()) == 1:\n            if is_main_process:\n                print(f"⏱️ [Stage B Time Budget Guard] Elapsed {time.time() - t_start:.1f}s >= {max_runtime_sec:.1f}s; finalizing Model Soup & Holdout Evaluation cleanly.", flush=True)\n            break\n\n    if is_main_process:\n        if student_resume_path.exists():\n            try:\n                student_resume_path.unlink()\n            except Exception:\n                pass\n        # Evaluate Top-3 Model Soup (Stochastic Weight Averaging) vs Best Single Checkpoint on Validation Set\n        best_model = BiometricMultiTaskModel(backbone_name="convnextv2_tiny", pretrained=False, drop_path_rate=0.0).to(device)\n        best_model.load_state_dict(torch.load(ckpt_path, map_location=device))\n        best_model.eval()\n        used_model_soup = False\n\n        if len(top_checkpoints) >= 2:\n            soup_state: Dict[str, torch.Tensor] = {}\n            K_soup = len(top_checkpoints)\n            for k, v0 in top_checkpoints[0][1].items():\n                if v0.dtype.is_floating_point:\n                    acc = torch.zeros_like(v0, dtype=torch.float32)\n                    for _, st in top_checkpoints:\n                        acc.add_(st[k].float(), alpha=1.0 / float(K_soup))\n                    soup_state[k] = acc.to(dtype=v0.dtype)\n                else:\n                    soup_state[k] = v0.clone()\n            soup_model = BiometricMultiTaskModel(backbone_name="convnextv2_tiny", pretrained=False, drop_path_rate=0.0).to(device)\n            soup_model.load_state_dict(soup_state)\n            soup_model.eval()\n            val_soup_m, _ = evaluate_dataset_regime(soup_model, val_ds, regime="clean", enable_flip_tta=True)\n            soup_score = val_soup_m["age_mae"] - 8.0 * val_soup_m["gender_accuracy"] - 5.0 * val_soup_m["fine_23_interval_within_1_bracket_acc"] - 3.0 * val_soup_m["fine_23_interval_exact_acc"]\n            if soup_score <= best_val_score:\n                torch.save(soup_state, ckpt_path)\n                best_model.load_state_dict(soup_state)\n                used_model_soup = True\n                print(f"🥣 [Model Soup] Top-{K_soup} SWA Checkpoint Soup improved Val Score ({soup_score:.4f} <= {best_val_score:.4f}) -> Saved to {ckpt_path}!", flush=True)\n\n        print(f"\\n🏆 Loaded Best Checkpoint from Epoch {best_epoch} (ModelSoup={used_model_soup}, {ckpt_path.stat().st_size / 1e6:.2f} MB). Running Holdout Test & Extreme-Noise Stress Benchmark...", flush=True)\n\n        val_clean_metrics, _ = evaluate_dataset_regime(best_model, val_ds, regime="clean", enable_flip_tta=True)\n        test_clean_metrics, test_clean_raw = evaluate_dataset_regime(best_model, test_ds, regime="clean", enable_flip_tta=True)\n\n        stress_results: Dict[str, Any] = {"clean": test_clean_metrics}\n        print("\\n=== HOLDOUT TEST EVALUATION ACROSS CLEAN & 6 EXTREME ENVIRONMENTAL NOISE REGIMES ===", flush=True)\n        print(f"  [clean                        ] Age MAE: {test_clean_metrics[\'age_mae\']:.2f}y | CS@5: {test_clean_metrics[\'cs_at_5\']*100:.1f}% | 23-Int(Exact/±1/±2): {test_clean_metrics[\'fine_23_interval_exact_acc\']*100:.1f}%/{test_clean_metrics[\'fine_23_interval_within_1_bracket_acc\']*100:.1f}%/{test_clean_metrics[\'fine_23_interval_within_2_brackets_acc\']*100:.1f}% | Gender Acc: {test_clean_metrics[\'gender_accuracy\']*100:.1f}% (AUC {test_clean_metrics[\'gender_roc_auc\']:.3f}) | Unc σ: {test_clean_metrics[\'mean_total_std\']:.2f}", flush=True)\n\n        for regime in ExtremeNoiseStressBenchmark.REGIMES[1:]:\n            reg_m, _ = evaluate_dataset_regime(\n                best_model, test_ds, regime=regime, enable_flip_tta=True, clean_ref_preds=test_clean_raw\n            )\n            stress_results[regime] = reg_m\n            print(\n                f"  [{regime:<29}] Age MAE: {reg_m[\'age_mae\']:.2f}y | CS@5: {reg_m[\'cs_at_5\']*100:.1f}% | "\n                f"23-Int(±1/±2): {reg_m[\'fine_23_interval_within_1_bracket_acc\']*100:.1f}%/{reg_m[\'fine_23_interval_within_2_brackets_acc\']*100:.1f}% | "\n                f"Gender Acc: {reg_m[\'gender_accuracy\']*100:.1f}% (AUC {reg_m[\'gender_roc_auc\']:.3f}) | "\n                f"Drift vs Clean: {reg_m[\'age_drift_vs_clean_years\']:.2f}y | Unc σ: {reg_m[\'mean_total_std\']:.2f}",\n                flush=True,\n            )\n\n        corrupted_regimes = [stress_results[r] for r in ExtremeNoiseStressBenchmark.REGIMES[1:]]\n        stress_summary = {\n            "mean_corrupted_age_mae": round(float(np.mean([m["age_mae"] for m in corrupted_regimes])), 4),\n            "mean_corrupted_cs_at_5": round(float(np.mean([m["cs_at_5"] for m in corrupted_regimes])), 4),\n            "mean_corrupted_23_interval_within_1_acc": round(float(np.mean([m["fine_23_interval_within_1_bracket_acc"] for m in corrupted_regimes])), 4),\n            "mean_corrupted_23_interval_within_2_acc": round(float(np.mean([m["fine_23_interval_within_2_brackets_acc"] for m in corrupted_regimes])), 4),\n            "mean_corrupted_gender_accuracy": round(float(np.mean([m["gender_accuracy"] for m in corrupted_regimes])), 4),\n            "mean_corrupted_gender_roc_auc": round(float(np.mean([m["gender_roc_auc"] for m in corrupted_regimes])), 4),\n            "mean_age_drift_vs_clean_years": round(float(np.mean([m["age_drift_vs_clean_years"] for m in corrupted_regimes])), 4),\n            "mean_gender_agreement_vs_clean": round(float(np.mean([m["gender_agreement_vs_clean"] for m in corrupted_regimes])), 4),\n            "uncertainty_monotonic_increase_on_noise": bool(\n                float(np.mean([m["mean_total_std"] for m in corrupted_regimes])) >= test_clean_metrics["mean_total_std"]\n            ),\n        }\n\n        fairness_report = evaluate_demographic_fairness(test_clean_raw)\n        calibration_report = evaluate_calibration_and_cohorts(test_clean_raw)\n        smoothness_latency_report = benchmark_smoothness_and_latency(best_model, test_ds)\n\n        src_col = next((c for c in ["source_dataset", "dataset_source", "dataset_name", "source", "dataset"] if c in train_ds.df.columns), None)\n        zero_utk = (int((train_ds.df[src_col].astype(str).str.lower() == "utkface").sum()) == 0) if src_col else True\n\n        full_report = {\n            "model_architecture": "ConvNeXtV2_Tiny_PhotometricRetinex_Hierarchical23Interval_KnowledgeDistilled_BiometricModel",\n            "backbone": "convnextv2_tiny (pretrained FCMAE + ImageNet-1K, distilled from convnextv2_base Teacher @ 224x224)",\n            "canonical_input_resolution": [224, 224, 3],\n            "dataset_manifest": str(manifest_p),\n            "splits": {\n                "train_samples": len(train_ds),\n                "val_samples": len(val_ds),\n                "test_samples": len(test_ds),\n                "zero_cross_split_subject_leakage": True,\n                "zero_utkface_records": zero_utk,\n            },\n            "knowledge_distillation_audit": {\n                "teacher_backbone": "convnextv2_base",\n                "teacher_params_millions": round(teacher_params_m, 2),\n                "student_backbone": "convnextv2_tiny",\n                "student_params_millions": round(sum(p.numel() for p in best_model.parameters()) / 1e6, 2),\n                "compression_ratio": round(teacher_params_m / max(1e-6, sum(p.numel() for p in best_model.parameters()) / 1e6), 2),\n                "teacher_epochs": teacher_epochs,\n                "refined_coarse_interval_samples": distill_prep_stats["refined_coarse_interval_samples"],\n                "teacher_imputed_missing_gender_samples": distill_prep_stats["teacher_imputed_missing_gender_samples"],\n                "used_top3_model_soup": used_model_soup,\n                "teacher_val_clean_metrics": teacher_val_metrics,\n                "teacher_test_clean_metrics": teacher_test_metrics,\n            },\n            "training": {\n                "num_epochs": num_epochs,\n                "best_epoch": best_epoch,\n                "batch_size": batch_size * world_size,\n                "runtime_seconds": round(time.time() - t_start, 2),\n                "checkpoint_size_mb": round(ckpt_path.stat().st_size / 1e6, 2),\n                "epoch_history": history,\n            },\n            "validation_clean_metrics": val_clean_metrics,\n            "test_clean_metrics": test_clean_metrics,\n            "calibration_and_per_source_audit": calibration_report,\n            "extreme_noise_stress_benchmark": stress_results,\n            "extreme_noise_summary": stress_summary,\n            "demographic_fairness_audit": fairness_report,\n            "smoothness_and_latency_benchmark": smoothness_latency_report,\n        }\n\n        report_path = Path("training_evaluation_report.json")\n        report_path.write_text(json.dumps(full_report, indent=2), encoding="utf-8")\n        print(f"\\n✅ Wrote comprehensive evaluation report to: {report_path.resolve()}", flush=True)\n        print(f"⏱️ Total Teacher+Student Training + Stress-Test Runtime: {full_report[\'training\'][\'runtime_seconds\']:.1f}s", flush=True)\n\n    if use_ddp and dist.is_initialized():\n        dist.barrier()\n\nif __name__ == "__main__":\n    try:\n        run_training_and_evaluation()\n    finally:\n        if dist.is_initialized():\n            dist.destroy_process_group()\n')
print('✅ Wrote train_ddp.py training and extreme-noise evaluation script.')


In [ ]:
import subprocess, sys, torch
num_gpus = torch.cuda.device_count()
if num_gpus >= 2:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={num_gpus}', 'train_ddp.py']
else:
    cmd = [sys.executable, '-u', 'train_ddp.py']
print(f'🚀 Launching: {" ".join(cmd)}')
proc = subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
import json
ckpt = Path('convnextv2_tiny_biometrics.pt')
rep = Path('training_evaluation_report.json')
assert ckpt.exists(), 'Checkpoint convnextv2_tiny_biometrics.pt missing!'
assert rep.exists(), 'Evaluation report training_evaluation_report.json missing!'
sz = ckpt.stat().st_size / 1e6
report_data = json.loads(rep.read_text(encoding='utf-8'))
print(f'🎉 Checkpoint ready: convnextv2_tiny_biometrics.pt ({sz:.2f} MB)')
print('📊 Clean Test Metrics:', json.dumps(report_data['test_clean_metrics'], indent=2))
print('🌪️ Extreme Noise Summary:', json.dumps(report_data['extreme_noise_summary'], indent=2))
print('⚡ Smoothness & Latency:', json.dumps(report_data['smoothness_and_latency_benchmark'], indent=2))
